# 02 · Data cleaning and storage
### Chess upsets over the board (DA2402 course project)

This notebook turns the tables written by 01_collection into the clean analysis table. Every rule is applied with its before/after count, and every removed game keeps its reason.

| Section | Content |
|---|---|
| 1 | inputs |
| 2 | time-control parser and its tests |
| 3 | time control estimated from the clocks |
| 4 | cleaning rules: results, variants, unplayed games, engines, duplicates, time control, ratings, names |
| 5 | why ratings are missing or disagree with FIDE |
| 6 | missing values: count, pattern, mechanism (tests), imputation or deletion |
| 7 | analysis filters and the derived columns of the clean table |
| 8 | outliers: detection (IQR, MAD, plausible range, Isolation Forest), diagnosis, decision |
| 9 | before and after cleaning |
| 10 | cleaning log |
| 11 | storage: final table, format comparison, dtypes after reading back, data dictionary |

**Run:** *Kernel > Restart & Run All* after notebook 01. On the full data (1.26 million games) the notebook needs about 10-15 minutes and about 4 GB of memory. Every section reads its inputs from disk, so a section can be re-run on its own after the configuration cells.

**Outputs:** `clean/analysis_games.parquet` (the final table); in `interim/`: `clock_features.parquet`, `clean_games.parquet` (games that pass the cleaning rules), `excluded_games.parquet` (every removed game with its reason), `analysis_derived.parquet`, `outlier_flags.parquet`; in `reports/`: `cleaning_log.csv`, `data_dictionary.csv`, `dtype_roundtrip.txt`, `format_comparison.csv`, `missingness_tests.csv`, `outlier_summary.csv`.

## 0. Configuration

In [1]:
import os

DATA_ROOT = os.environ.get("CHESS_DATA_ROOT", os.path.join("..", "data"))   # holds raw/ interim/ clean/
REPORTS   = os.environ.get("CHESS_REPORTS", os.path.join("..", "reports"))
RAW, INTERIM, CLEAN = (os.path.join(DATA_ROOT, d) for d in ("raw", "interim", "clean"))
FIDE_FIRST, FIDE_LAST = "2020-01", "2026-09"     # months of the FIDE lists downloaded in notebook 01
SEED = 42

In [2]:
import re, glob, time, sqlite3, tempfile, warnings, datetime, platform
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 120)
def say(text): display(Markdown(text))                     # prose whose numbers are computed from the data
os.makedirs(CLEAN, exist_ok=True); os.makedirs(REPORTS, exist_ok=True)

ALL_GAMES     = os.path.join(INTERIM, "all_games.parquet")
FIDE_RATINGS  = os.path.join(INTERIM, "fide_ratings.parquet")
FIDE_PLAYERS  = os.path.join(INTERIM, "fide_players.parquet")
CLOCK         = os.path.join(INTERIM, "clock_features.parquet")
CLEAN_GAMES   = os.path.join(INTERIM, "clean_games.parquet")
EXCLUDED      = os.path.join(INTERIM, "excluded_games.parquet")
DERIVED       = os.path.join(INTERIM, "analysis_derived.parquet")
FLAGS         = os.path.join(INTERIM, "outlier_flags.parquet")
FINAL         = os.path.join(CLEAN, "analysis_games.parquet")
TC = ["classical", "rapid", "blitz"]

def read_columns(path, cols):
    """Read the listed columns of a Parquet file; a column the file does not have comes back as all missing."""
    have = set(pq.ParquetFile(path).schema_arrow.names)
    df = pq.read_table(path, columns=[c for c in cols if c in have]).to_pandas()
    for c in cols:
        if c not in df: df[c] = pd.Series(pd.NA, index=df.index, dtype="object")
    return df[cols]

def write_parquet(df, path):                               # temporary file first: an interrupted run leaves no broken file
    df.to_parquet(path + ".tmp", index=False); os.replace(path + ".tmp", path)

def write_rows(src, out, keep, extra):
    """Copy the rows of Parquet file `src` where `keep` is True (file order) to `out`, adding or replacing the columns of
    DataFrame `extra` (one row per kept row, same order). Works batch by batch, so the wide table is never fully in memory."""
    keep = np.asarray(keep, dtype=bool); extra = extra.reset_index(drop=True)
    assert keep.sum() == len(extra), "extra must have one row per kept row"
    pf = pq.ParquetFile(src)
    raw_fields = [f for f in pf.schema_arrow if f.name not in extra.columns]
    extra_schema = pa.Schema.from_pandas(extra, preserve_index=False).remove_metadata()
    schema = pa.schema(raw_fields + list(extra_schema))
    start = pos = 0
    with pq.ParquetWriter(out + ".tmp", schema) as w:
        for b in pf.iter_batches(batch_size=200_000, columns=[f.name for f in raw_fields]):
            idx = np.flatnonzero(keep[start:start + b.num_rows]); start += b.num_rows
            if not len(idx): continue
            t = pa.Table.from_batches([b]).take(pa.array(idx))
            e = pa.Table.from_pandas(extra.iloc[pos:pos + len(idx)], schema=extra_schema, preserve_index=False); pos += len(idx)
            for f in extra_schema: t = t.append_column(f, e.column(f.name))
            w.write_table(t)
    os.replace(out + ".tmp", out)

def kind(x): return "string" if x.dtype == object or pd.api.types.is_string_dtype(x) else str(x.dtype)
LOG = []                                                   # one entry per cleaning step; saved per section, combined in section 10
def log_step(stage, step, action, rows, left):
    LOG.append(dict(stage=stage, step=step, action=action, rows_affected=int(rows), rows_left=int(left)))
def save_log(part):
    pd.DataFrame(LOG).to_csv(os.path.join(INTERIM, f"_log_{part}.csv"), index=False); LOG.clear()
print(f"python {platform.python_version()} | pandas {pd.__version__} | pyarrow {pa.__version__} | run on {datetime.date.today()}")
print("DATA_ROOT =", os.path.abspath(DATA_ROOT))

python 3.13.5 | pandas 3.1.0 | pyarrow 26.0.0 | run on 2026-11-05
DATA_ROOT = C:\dev\chess-upsets\submission\data


## 1. Inputs

The tables written by notebook 01 (unchanged raw values). Their shape, data types and the meaning of every column are shown there (section 7).

In [3]:
inputs = [ALL_GAMES, FIDE_RATINGS, FIDE_PLAYERS] + sorted(glob.glob(os.path.join(INTERIM, "moves_*.parquet")))
info = pd.DataFrame([(os.path.basename(p), pq.ParquetFile(p).metadata.num_rows, pq.ParquetFile(p).metadata.num_columns, round(os.path.getsize(p) / 1e6, 1))
                     for p in inputs], columns=["file", "rows", "columns", "MB"])
n_moves = len(info) - 3
display(pd.concat([info.head(3), pd.DataFrame([dict(file=f"moves_*.parquet ({n_moves} months)", rows=info.rows[3:].sum(), columns=2, MB=info.MB[3:].sum())])], ignore_index=True))

,file,rows,columns,MB
0,all_games.parquet,1263916,52,118.7
1,fide_ratings.parquet,4994547,14,45.3
2,fide_players.parquet,88503,6,2.3
3,moves_*.parquet (81 months),1263916,2,1556.8


## 2. Time-control parser

The `TimeControl` tag is missing for more than half of the games, and where present it is written in hundreds of ways: `90+30`, `5400+30`, `1800+3`, `90min+30sec/move`, `40/5400+30:1800`, `G90' + 30 per move`, `1:30:00 +30`, etc. `parse_tc` returns base time and increment in seconds and the name of the rule that matched. Number rule: a number up to 120 is read as minutes, from 300 as seconds; 121-299 is ambiguous and read as seconds (method `numeric_ambiguous`). Clock notation `h:mm`, `h:mm:ss` or `h:mm.ss` (e.g. `1:30:00 +30`, `1:30.00 + 0.30 increment`) is read as hours and minutes, the increment after `+` as seconds (`0.30` = 0 min 30 s); a bare `1:30:30` is 1 h 30 min + 30 s (method `text_hms`); every parse is checked against the game's clock in section 4.6. `tc_class` applies the FIDE definition: total = base + 60 × increment; blitz ≤ 600 s, rapid < 3,600 s, classical ≥ 3,600 s.

In [4]:
NUM    = re.compile(r"^\s*(\d+)\s*\+\s*(\d+)\s*$")                    # 90+30, 900+10
SINGLE = re.compile(r"^\s*(\d+)\s*$")                                  # 600, 1500, 90
STAGED = re.compile(r"^\s*(\d+)\s*/\s*(\d+)\s*\+\s*(\d+)\s*:")         # 5400/40+30:1800, 40/5400+30:1800
TRIPLE = re.compile(r"^\s*(\d+)\s*\+\s*(\d+)\s*\+\s*(\d+)\s*$")        # 5400+30+1800
COLON  = re.compile(r"^\s*(\d+)\s*:\s*(\d+)\s*$")                      # 5400:30
MIN    = re.compile(r"(\d+)\s*(?:min\w*|m\b|mn\b|')", re.I)
SEC    = re.compile(r"(\d+)\s*(?:sec\w*|seg\w*|sg\b|s\b|\")", re.I)
GAME   = re.compile(r"\bg\s*:?\s*(\d+)", re.I)                         # "G90' + 30", "G: 20 minutes"
LOOSE  = re.compile(r"(\d+)\s*[+´`']\s*(\d+)")                         # '90+30´ 30', '900 + 5 sec'
HMS    = re.compile(r"^\s*(\d):([0-5]\d)(?:([:.])([0-5]\d))?(?![\d:])")  # clock notation h:mm, h:mm:ss, h:mm.ss: '1:30:00 +30', '1:30.00 + 30 seconds'
HMS_INC = re.compile(r"\+\s*(\d+)(?:[.:]([0-5]\d))?")                   # its increment: '+ 30' (seconds) or '+ 0.30' / '+ 0:30' (m.ss)

def _num_rule(a, b, tag="numeric"):
    if a <= 120: return a * 60, b, f"{tag}_min"
    if a >= 300: return a, b, f"{tag}_sec"
    return a, b, f"{tag}_ambiguous"

def parse_tc(s):
    """TimeControl string -> (base seconds, increment seconds, method)."""
    if s is None or s != s or str(s).strip() in ("", "-", "?"):
        return None, None, "missing"
    s = str(s).strip()
    m = NUM.match(s)
    if m: return _num_rule(int(m.group(1)), int(m.group(2)))
    m = SINGLE.match(s)
    if m: return _num_rule(int(m.group(1)), 0)
    m = STAGED.match(s)
    if m:
        x, y, z = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if x <= 60 and y >= 300: return y, z, "staged"
        if y <= 60 and x >= 300: return x, z, "staged"
        return None, None, "unparsed"
    m = TRIPLE.match(s)
    if m: return _num_rule(int(m.group(1)), int(m.group(2)))
    m = COLON.match(s)
    if m and int(m.group(1)) >= 300: return int(m.group(1)), int(m.group(2)), "numeric_sec"
    m = HMS.match(s)
    if m:
        h, mi, sep, ss = int(m.group(1)), int(m.group(2)), m.group(3), int(m.group(4) or 0)
        im = HMS_INC.search(s, m.end())
        if im is None and sep == ":" and ss > 0:          # '1:30:30' = 1 h 30 min + 30 s increment (confirmed by the clocks of these games)
            return h * 3600 + mi * 60, ss, "text_hms"
        inc = 0 if im is None else (int(im.group(1)) * 60 + int(im.group(2)) if im.group(2) else int(im.group(1)))
        return h * 3600 + mi * 60 + ss, inc, "text_hms"
    base = inc = None
    mm, ms = MIN.search(s), SEC.search(s)
    if mm: base = int(mm.group(1)) * 60
    else:
        gm = GAME.search(s)
        if gm: base = int(gm.group(1)) * 60
    if ms: inc = int(ms.group(1))
    if base is not None and inc is None:
        after = re.search(r"\+\s*(\d+)", s)
        if after: inc = int(after.group(1))
    if base is None:
        lm = LOOSE.search(s)
        if lm:
            b, i, _ = _num_rule(int(lm.group(1)), int(lm.group(2)), "text_loose")
            return b, i, "text_loose"
        return None, None, "unparsed"
    return base, (inc or 0), "text"

def tc_class(base, inc):
    """FIDE: blitz <= 10 min, rapid 10..60 min, classical >= 60 min, where total = base + 60 * increment."""
    if base is None or base != base: return None
    total = base + 60 * (inc or 0)
    return "blitz" if total <= 600 else ("rapid" if total < 3600 else "classical")

**Tests.** The cases below were collected from the data (every spelling that once failed or was mis-parsed on the full 81 months is included); the cell stops if any case is wrong.

In [5]:
CASES = [("90+30",5400,30,"classical"), ("3+2",180,2,"blitz"), ("60+30",3600,30,"classical"), ("15+10",900,10,"rapid"), ("10+5",600,5,"rapid"),
 ("1800+3",1800,3,"rapid"), ("45+10",2700,10,"rapid"), ("20+5",1200,5,"rapid"), ("10+3",600,3,"rapid"), ("25+10",1500,10,"rapid"),
 ("60+15",3600,15,"classical"), ("5400+30",5400,30,"classical"), ("30+30",1800,30,"classical"), ("900+10",900,10,"rapid"), ("10+10",600,10,"rapid"),
 ("5+3",300,3,"blitz"), ("2700+15",2700,15,"classical"), ("15+5",900,5,"rapid"), ("20+10",1200,10,"rapid"),
 ("90 minutes for the first 40 moves, followed by 30 minutes for the rest of the game with anincrement of 30 seconds per move starting from move 1",5400,30,"classical"),
 ("90min+30sec/move",5400,30,"classical"), ("90 minutes with 30 second increment from move 1",5400,30,"classical"), ("G90' + 30 per move",5400,30,"classical"),
 ("90 min + 30 sec",5400,30,"classical"), ("Standard: 60 minutes with 30 seconds increment from move 1",3600,30,"classical"), ("45m+10s",2700,10,"rapid"),
 ("90 minutes + 30 seconds per move",5400,30,"classical"), ("45 mininutes for the game and 10 seconds increment from move 1",2700,10,"rapid"),
 ("10 min + 5 sec/move Fischer",600,5,"rapid"), ("90min + 30 sg",5400,30,"classical"), ("90 min + 30 seconds permove from move number one",5400,30,"classical"),
 ("8'+ 3",480,3,"rapid"), ("90+30´ 30",5400,30,"classical"), ("90 min / 40 moves + 30 min + 30 sec / move",5400,30,"classical"), ("10 min + 5 seg add",600,5,"rapid"),
 ("G: 20 minutes + 5 seconds increment from move 1",1200,5,"rapid"), ("90min+30sec",5400,30,"classical"), ("90 min + 30 sec 40/5",5400,30,"classical"),
 ("90 min + 30 sg",5400,30,"classical"), ("600",600,0,"blitz"), ("1500",1500,0,"rapid"), ("3600",3600,0,"classical"), ("10800",10800,0,"classical"),
 ("5400/40+30:1800+30\\\"",5400,30,"classical"), ("5400/40+30:1800",5400,30,"classical"), ("40/5400+30:1800",5400,30,"classical"),
 ("5400+30+1800",5400,30,"classical"), ("5400:30",5400,30,"classical"), ("900 + 5 sec",900,5,"rapid"), ("180+2",180,2,"blitz"),
 ("1:30:00 +30",5400,30,"classical"), ("1:30:00+30",5400,30,"classical"), ("1:30.00 + 30 seconds increment from move 1",5400,30,"classical"),
 ("1:00.00 + 0.30 increment from move 1",3600,30,"classical"), ("1:30.00 f.b. 0:30.00 + 0.30 increment from move 1",5400,30,"classical"),
 ("1:30:30",5400,30,"classical"), ("1:30 + 30sec inc",5400,30,"classical"), ("1:50 + 0.10",6600,10,"classical"), ("1:30:00",5400,0,"classical")]
wrong = [(s, parse_tc(s), tc_class(*parse_tc(s)[:2])) for s, b, i, c in CASES if (*parse_tc(s)[:2], tc_class(*parse_tc(s)[:2])) != (b, i, c)]
assert not wrong, wrong
say(f"All **{len(CASES)}** test strings are parsed correctly; missing and unreadable strings give {parse_tc(None)} and {parse_tc('blah')}.")

All **59** test strings are parsed correctly; missing and unreadable strings give (None, None, 'missing') and (None, None, 'unparsed').

## 3. Time control estimated from the clocks

Most broadcast games carry the players' clock after every move. From these readings the time control can be estimated even when the tag is missing:

* `inc_hat` = the largest clock increase between two consecutive moves of the same player within the first 30 moves of each side (before any move-40 time bonus); jumps above 600 s are ignored as clock glitches;
* `base_hat` = White's first clock reading − `inc_hat` + 1 (after the first move the clock shows base + increment − time spent); a value outside 30 s to 4 h is discarded.

The estimate is computed per month (`interim/clock_features/clock_YYYY-MM.parquet`, skipped when up to date) and combined into `interim/clock_features.parquet`.

In [6]:
MOVE = re.compile(r'(\d+)(\.{1,3})\s*([^\s{]+)((?:\s*\{[^}]*\})*)')
CLK  = re.compile(r'\[%clk (\d+):(\d+):(\d+(?:\.\d+)?)\]')
def clk_seconds(comments):
    m = CLK.search(comments)
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3)) if m else None

def estimate(text):
    """(base_hat, inc_hat, number of clock readings) from the move text of one game."""
    if not isinstance(text, str) or "%clk" not in text:
        return None, None, 0
    w, b = [], []
    for num, dots, san, com in MOVE.findall(text):
        if com:
            c = clk_seconds(com)
            if c is not None: (w if dots == "." else b).append(c)
    deltas = []
    for s in (w[:30], b[:30]):
        deltas += [s[i] - s[i - 1] for i in range(1, len(s))]
    pos = [d for d in deltas if 0 < d <= 600]
    inc = max(pos) if pos else 0.0
    first = w[0] if w else (b[0] if b else None)
    base = first - inc + 1 if first is not None else None
    if base is not None and not (30 <= base <= 14400):
        base = None
    return base, inc, len(w) + len(b)

In [7]:
months = sorted(pq.read_table(ALL_GAMES, columns=["source_month"]).column(0).unique().to_pylist())
os.makedirs(os.path.join(INTERIM, "clock_features"), exist_ok=True)
t0, status = time.time(), []
for m in months:
    src, out = os.path.join(INTERIM, f"moves_{m}.parquet"), os.path.join(INTERIM, "clock_features", f"clock_{m}.parquet")
    if os.path.exists(out) and os.path.getmtime(out) >= os.path.getmtime(src):
        status.append("up to date"); continue
    mv = pd.read_parquet(src)
    est = [estimate(t) for t in mv["movetext"]]
    write_parquet(pd.DataFrame({"game_url": mv["game_url"], "base_hat": [e[0] for e in est], "inc_hat": [e[1] for e in est],
                                "n_clk": [e[2] for e in est]}).astype({"base_hat": "float64", "inc_hat": "float64"}), out)
    status.append("computed now")
cf = pd.concat([pd.read_parquet(os.path.join(INTERIM, "clock_features", f"clock_{m}.parquet")) for m in months], ignore_index=True)
write_parquet(cf, CLOCK)
say(f"Clock estimates for **{len(cf):,}** games from {len(months)} months ({status.count('computed now')} computed now, {status.count('up to date')} up to date; {time.time() - t0:.0f} s). "
    f"A base time could be estimated for **{cf.base_hat.notna().sum():,}** games ({100 * cf.base_hat.notna().mean():.1f} %).")
del cf

Clock estimates for **1,263,916** games from 81 months (0 computed now, 81 up to date; 1 s). A base time could be estimated for **1,127,314** games (89.2 %).

## 4. Cleaning rules

The raw games are loaded once (only the columns the rules need). Each game is removed for the first rule that applies, in the order below, and keeps that rule as its `reason` (`interim/excluded_games.parquet`), so nothing disappears silently. Two fields that are messy but recoverable are repaired rather than removed: the time control (4.6) and the ratings (4.7).

In [8]:
WORK = ["game_url", "result", "variant", "fen", "set_up", "n_plies", "termination", "white_elo", "black_elo", "broadcast_name",
        "broadcast_url", "round", "white_fide_id", "black_fide_id", "time_control", "first_clk_w", "first_clk_b", "game_date", "source_month"]
g = read_columns(ALL_GAMES, WORK)
for c in ("white_elo", "black_elo"):
    g[c] = pd.to_numeric(g[c], errors="coerce").astype("Int64")
cf = pd.read_parquet(CLOCK).drop_duplicates("game_url")
g = g.merge(cf[["game_url", "base_hat", "inc_hat"]], on="game_url", how="left")     # keeps the row order of all_games
del cf
N0 = len(g)
assert N0 == pq.ParquetFile(ALL_GAMES).metadata.num_rows
def col(name): return g[name] if name in g.columns else pd.Series(pd.NA, index=g.index, dtype="object")

g["reason"] = pd.Series(None, index=g.index, dtype="object")
def drop(mask, name):
    """Remove the games in mask that are not removed yet; log the count."""
    mask = mask.fillna(False).astype(bool)
    new = mask & g["reason"].isna()
    g.loc[new, "reason"] = name
    log_step("1 cleaning rules", name, "remove", new.sum(), g["reason"].isna().sum())
    return int(new.sum())
log_step("0 collection", "all games parsed (notebook 01)", "start", N0, N0)
say(f"Raw table: **{N0:,}** games.")

Raw table: **1,263,916** games.

### 4.1 Result strings

The `Result` tag should be `1-0`, `0-1` or `1/2-1/2`. Spelling variants (`½-½`, `1:0`, `0.5-0.5`, etc.) are normalised and mapped to White's score; unfinished games (`*`), forfeit codes (`+--`, `1F-0F`, etc.) and invalid strings have no score and are removed.

In [9]:
PLAYED = {"1-0": 1.0, "0-1": 0.0, "1/2-1/2": 0.5, "0.5-0.5": 0.5}
FORFEIT = {"+--", "--+", "+/-", "-/+", "-+", "+-"}
def norm_result(s):
    if s == "<NA>": return "invalid", np.nan
    t = (str(s).strip().replace("½", "1/2").replace("–", "-").replace("\u2014", "-")
         .replace(" ", "").replace(":", "-").replace("_", "-").lower())
    if t in PLAYED: return "played", PLAYED[t]
    if t == "*": return "unfinished", np.nan
    if t in FORFEIT or re.fullmatch(r"[01]f-[01]f", t): return "forfeit", np.nan
    return "invalid", np.nan
r = g["result"].astype("object").where(g["result"].notna(), "<NA>")
rmap = {u: norm_result(u) for u in r.unique()}
rkind = r.map(lambda u: rmap[u][0])
g["score_white"] = r.map(lambda u: rmap[u][1]).astype(float)
vc = r.value_counts()
before = vc.rename("games").to_frame().assign(rule=lambda d: [rmap[u][0] for u in d.index], score_white=lambda d: [rmap[u][1] for u in d.index])
n_respelled = int(((rkind == "played") & ~r.isin(["1-0", "0-1", "1/2-1/2"])).sum())
removed = {k: drop(rkind == k, name) for k, name in (("unfinished", "1 unfinished result (*)"), ("forfeit", "2 forfeit result code"), ("invalid", "3 invalid result string"))}
log_step("2 fixes", "result spelling normalised (e.g. ½-½ -> 1/2-1/2)", "fix", n_respelled, g["reason"].isna().sum())
say(f"**Before:** {len(vc)} distinct result strings, of which {sum(1 for u in vc.index if rmap[u][0] == 'played')} are played results. "
    f"**After:** {int((rkind == 'played').sum()):,} played games keep a score ({n_respelled:,} of them after normalising the spelling); "
    f"{removed['unfinished']:,} unfinished, {removed['forfeit']:,} forfeit codes and {removed['invalid']:,} invalid strings are removed.")
display(before.head(12))

**Before:** 25 distinct result strings, of which 11 are played results. **After:** 1,173,194 played games keep a score (84 of them after normalising the spelling); 90,666 unfinished, 29 forfeit codes and 27 invalid strings are removed.

,games,rule,score_white
result,,,
1-0,477920,played,1.0
0-1,397409,played,0.0
1/2-1/2,297781,played,0.5
*,90666,unfinished,NaN
½-½,72,played,0.5
0-0,16,invalid,NaN
--+,9,forfeit,NaN
+--,6,forfeit,NaN
–/+,5,forfeit,NaN


### 4.2 Variants and non-standard start positions

The question is about standard chess. Games with another variant (e.g. Chess960) or a set-up start position (`FEN` / `SetUp "1"`) are removed.

In [10]:
var = col("variant").astype("string").str.lower()
nonstd = (var.notna() & (var != "standard")) | col("fen").notna() | (col("set_up").astype("string") == "1")
display(col("variant").astype("object").where(col("variant").notna(), "<missing>").value_counts().head(6).rename("games").to_frame())
n = drop(nonstd, "4 non-standard variant / Chess960 / start position")
say(f"**{int(nonstd.sum()):,}** games are not standard chess from the initial position; **{n:,}** of them are removed here (the others were already removed above).")

,games
variant,
Standard,1252259
Chess960,5577
From Position,4407
Fischerandom,1147
fischerandom,425
Crazyhouse,59


**13,860** games are not standard chess from the initial position; **13,382** of them are removed here (the others were already removed above).

### 4.3 Unplayed games

A game with no moves, or with termination `Unplayed` / `Unterminated`, was not played (a forfeit or a board that was never relayed).

In [11]:
term = col("termination").astype("string").str.lower()
unplayed = (g["n_plies"] == 0) | term.isin(["unplayed", "unterminated"])
n = drop(unplayed, "5 no moves / unplayed (incl. forfeits)")
say(f"Games with no moves: **{int((g['n_plies'] == 0).sum()):,}**; with termination Unplayed/Unterminated: **{int(term.isin(['unplayed', 'unterminated']).sum()):,}**. Removed here: **{n:,}**.")

Games with no moves: **89,318**; with termination Unplayed/Unterminated: **6,972**. Removed here: **21,465**.

### 4.4 Engines and adjudicated games

Ratings between 2,900 and 4,000 are engine ratings (no human is rated above about 2,880): such games are computer games. Adjudicated games were not played to the end. Events in which at least half of the games are engine or adjudicated games are removed as a whole (exhibition and engine events).

In [12]:
we_raw, be_raw = col("white_elo"), col("black_elo")
eng_elo = ((we_raw > 2900) & (we_raw <= 4000)) | ((be_raw > 2900) & (be_raw <= 4000))
adj = term.str.startswith("adjudication").fillna(False)
bshare = (eng_elo | adj).groupby(g["broadcast_name"]).transform("mean").fillna(0)
n6, n7, n8 = drop(eng_elo, "6 engine-level Elo (2900-4000)"), drop(adj, "7 adjudicated game"), drop(bshare >= 0.5, "8 engine/adjudicated event (>=50% of its games flagged)")
say(f"Removed: **{n6:,}** games with an engine-level rating, **{n7:,}** adjudicated games, and **{n8:,}** further games of events where at least half of the games are flagged.")

Removed: **8,383** games with an engine-level rating, **1** adjudicated games, and **116** further games of events where at least half of the games are flagged.

### 4.5 Duplicates

Two rules, applied to the games that survived the earlier steps:

1. Exact duplicate: the same `game_url` (the unique game key) appears twice, for example in two monthly files.
2. The same game under two records (duplicate across rounds, studies or relays): the same two FIDE ids, round, broadcast round URL, result and number of half-moves. Player names are not used because of the spelling variation shown in 4.8; games without both FIDE ids are not checked by this rule.

The first record is kept. Across the two sources, duplicates in the FIDE lists are handled in 4.7 (one row per FIDE id and month).

In [13]:
cand1 = int(g.game_url.duplicated().sum())
rem = g[g["reason"].isna()]
dup = rem.index[rem.duplicated("game_url", keep="first")]
n9 = drop(pd.Series(g.index.isin(dup), index=g.index), "9 duplicate game_url")
KEY = ["white_fide_id", "black_fide_id", "round", "broadcast_url", "score_white", "n_plies"]
cand2 = int(g.dropna(subset=["white_fide_id", "black_fide_id"]).duplicated(KEY).sum())
rem = g[g["reason"].isna() & col("white_fide_id").notna() & col("black_fide_id").notna()]
dup = rem.index[rem.duplicated(KEY, keep="first")]
n10 = drop(pd.Series(g.index.isin(dup), index=g.index), "10 duplicate (same players, round, broadcast, result, length)")
say(f"In the raw table **{cand1:,}** games repeat a `game_url` and **{cand2:,}** repeat the rule-2 key. After the earlier steps **{n9:,}** exact and **{n10:,}** "
    f"rule-2 duplicates remain and are removed; the other candidates had already been removed for another reason.")

In the raw table **4** games repeat a `game_url` and **510** repeat the rule-2 key. After the earlier steps **0** exact and **315** rule-2 duplicates remain and are removed; the other candidates had already been removed for another reason.

### 4.6 Time control: string, checked against the clock

1. The string is parsed (section 2).
2. Where the game also has a clock, the parse is verified: base + increment must lie within 2 minutes of the first clock reading.
3. Where the string is missing or disagrees with the clock, the clock estimate (section 3) is used. The estimate is noisy by a few seconds of increment, and the class depends on base + 60 × increment, so it is snapped to the nearest time control that was verified on at least 200 games (e.g. 5400+30) if one lies close enough.
4. Where neither works, an event name containing exactly one of *rapid*, *blitz*, *classical/standard* gives the class.
5. Otherwise the time control is `unknown`.

In [14]:
tcs = g["time_control"].astype("object").where(g["time_control"].notna(), None)
tmap = {u: parse_tc(u) for u in pd.unique(tcs)}
sb = pd.to_numeric(tcs.map(lambda u: tmap[u][0]), errors="coerce")
si = pd.to_numeric(tcs.map(lambda u: tmap[u][1]), errors="coerce")
g["tc_method"] = tcs.map(lambda u: tmap[u][2])
string_ok = sb.notna() & ~g["tc_method"].isin(["missing", "unparsed"])
clk0 = pd.to_numeric(g["first_clk_w"], errors="coerce").fillna(pd.to_numeric(g["first_clk_b"], errors="coerce"))
agrees = string_ok & clk0.notna() & ((clk0 - (sb + si)).abs() <= 120)
use_string = string_ok & (agrees | clk0.isna())

def build_snapper(pairs, min_count=200):
    known = [(b, i, c) for (b, i), c in pairs.items() if c >= min_count]
    def snap(b, i):
        best = None
        for cb, ci, c in known:
            if abs(i - ci) <= 3 and abs(b - cb) <= max(90, 0.04 * cb):
                if best is None or c > best[2]: best = (cb, ci, c)
        return (best[0], best[1]) if best else None
    return snap
def apply_snap(snap, base_hat, inc_hat):
    keys = pd.DataFrame({"b": base_hat.round(), "i": inc_hat.round()})
    uk = keys.dropna().drop_duplicates().reset_index(drop=True)
    res = [snap(b, i) for b, i in zip(uk.b, uk.i)]
    uk["sb"] = [r[0] if r else np.nan for r in res]; uk["si"] = [r[1] if r else np.nan for r in res]
    out = keys.merge(uk, on=["b", "i"], how="left"); out.index = base_hat.index
    return out["sb"], out["si"]
def klass(base, inc):
    total = base + 60 * inc
    return pd.Series(np.select([total <= 600, total < 3600], ["blitz", "rapid"], "classical"), index=base.index).where(base.notna())

snap_all = build_snapper(pd.DataFrame({"b": sb[agrees], "i": si[agrees]}).value_counts())
snap_b, snap_i = apply_snap(snap_all, g["base_hat"], g["inc_hat"])
is_snapped = snap_b.notna()
base_c, inc_c = snap_b.fillna(g["base_hat"]), snap_i.fillna(g["inc_hat"])
use_clock = ~use_string & base_c.notna()
g["tc_base_s"] = np.where(use_string, sb, np.where(use_clock, base_c, np.nan))
g["tc_inc_s"] = np.where(use_string, si, np.where(use_clock, inc_c, np.nan))
g["tc_class"] = klass(g["tc_base_s"], g["tc_inc_s"])
g["tc_source"] = np.select([use_string & agrees, use_string & ~agrees, use_clock & string_ok, use_clock & is_snapped, use_clock],
                           ["string_verified_by_clock", "string_unverified(no clock)", "clock_override", "clock_snapped", "clock_raw"], "none")
_tot = g["tc_base_s"] + 60 * g["tc_inc_s"]
g["tc_borderline"] = (np.minimum((_tot - 600).abs(), (_tot - 3600).abs()) <= 180)
name = col("broadcast_name").astype("string").str.lower().fillna("")
hr, hb, hc = name.str.contains("rapid"), name.str.contains("blitz"), name.str.contains("classical|klassisch|classic|standard")
only = (hr.astype(int) + hb.astype(int) + hc.astype(int)) == 1
need = g["tc_class"].isna()
g.loc[need & only & hr, "tc_class"] = "rapid"
g.loc[need & only & hb, "tc_class"] = "blitz"
g.loc[need & only & hc, "tc_class"] = "classical"
g.loc[need & only & g["tc_class"].notna(), "tc_source"] = "event_name"
g["tc_class"] = g["tc_class"].fillna("unknown")

kept = g["reason"].isna()
n_str_missing = int(tcs.isna().sum())
say(f"**Before:** the string is missing for **{n_str_missing:,} games ({100 * n_str_missing / N0:.1f} %)**; the rest uses **{tcs.dropna().nunique():,} distinct spellings**. "
    f"**After** (games that pass rules 1–10): a time control is known for **{100 * (g.loc[kept, 'tc_class'] != 'unknown').mean():.1f} %**.")
nice = {"string_verified_by_clock": "string, confirmed by the game's first clock", "string_unverified(no clock)": "string, no clock to confirm it",
        "clock_snapped": "clock estimate, snapped to a verified time control", "clock_raw": "clock estimate (no usable string)",
        "clock_override": "clock replaced a string that disagreed with it", "event_name": "class from the event name", "none": "unknown"}
src = g.loc[kept, "tc_source"].value_counts()
display(src.rename(index=nice).rename("games").to_frame().assign(share_pct=lambda d: (100 * d.games / d.games.sum()).round(1)))
for s_, label, act in (("string_verified_by_clock", "time control parsed from the string and confirmed by the clock", "check"),
                       ("clock_override", "time control string replaced by the clock estimate (they disagreed)", "fix"),
                       ("clock_snapped", "time control estimated from the clock, snapped to a verified time control", "fix"),
                       ("clock_raw", "time control estimated from the clock", "fix"), ("event_name", "time-control class taken from the event name", "fix"),
                       ("none", "time control unknown", "flag")):
    log_step("2 fixes", label, act, src.get(s_, 0), kept.sum())

**Before:** the string is missing for **703,934 games (55.7 %)**; the rest uses **995 distinct spellings**. **After** (games that pass rules 1–10): a time control is known for **97.4 %**.

,games,share_pct
tc_source,,
"string, confirmed by the game's first clock",435415,38.5
"clock estimate, snapped to a verified time control",365823,32.4
clock estimate (no usable string),233776,20.7
clock replaced a string that disagreed with it,50648,4.5
unknown,29717,2.6
"string, no clock to confirm it",11023,1.0
class from the event name,3130,0.3


**Check of the clock estimate.** Where both the string and the clock give a class, the two are compared. The snapping is checked out of sample: the verified time controls are learned from the odd months only and the snapped class is compared with the string on the even months.

In [15]:
cs = klass(sb, si).where(string_ok)
cc = klass(g["base_hat"], g["inc_hat"])
both = cs.notna() & cc.notna() & kept
if both.any():
    say(f"Class from the string and from the raw clock estimate agree for **{100 * (cs[both] == cc[both]).mean():.1f} %** of the {int(both.sum()):,} games that have both:")
    display(pd.crosstab(cs[both], cc[both], rownames=["string"], colnames=["clock"]))
mon = pd.to_numeric(g["source_month"].astype(str).str[-2:], errors="coerce")
learn, test = agrees & (mon % 2 == 1), agrees & (mon % 2 == 0)
snap_odd = build_snapper(pd.DataFrame({"b": sb[learn], "i": si[learn]}).value_counts())
tb, ti = apply_snap(snap_odd, g["base_hat"][test], g["inc_hat"][test])
cc_snap_test = klass(tb.fillna(g["base_hat"][test]), ti.fillna(g["inc_hat"][test]))
kt = cs[test & kept].index
kt = kt[cc_snap_test.reindex(kt).notna().to_numpy() & cc.reindex(kt).notna().to_numpy()]
if len(kt):
    say(f"Out of sample (even months, n = {len(kt):,}): class agreement **{100 * (cs[kt] == cc[kt]).mean():.1f} %** for the raw clock estimate and "
        f"**{100 * (cs[kt] == cc_snap_test[kt]).mean():.1f} %** after snapping. Disagreements are mostly games near a class boundary; "
        f"they are flagged `tc_borderline` (total time within 3 minutes of 10 or 60 minutes: {100 * g.loc[kept, 'tc_borderline'].mean():.1f} % of games).")
else:
    say("Out-of-sample check skipped: the data hold no verified games in even months.")

Class from the string and from the raw clock estimate agree for **96.1 %** of the 485,725 games that have both:

clock,blitz,classical,rapid
string,,,
blitz,74657,532,2102
classical,345,283780,10056
rapid,728,5016,108509


Out of sample (even months, n = 224,110): class agreement **96.9 %** for the raw clock estimate and **97.4 %** after snapping. Disagreements are mostly games near a class boundary; they are flagged `tc_borderline` (total time within 3 minutes of 10 or 60 minutes: 4.4 % of games).

### 4.7 Ratings: validate, then fill from the FIDE list

* A PGN rating is valid if it lies between 1,000 and 2,900 (FIDE's rating floor is 1,000 and no player has ever been rated above about 2,880); other values are set to missing (they are placeholders such as 0 or typing errors; 2,900-4,000 were removed in 4.4 as engines).
* A missing rating is filled from the FIDE list of the game's month and of the game's time control (standard list for classical, rapid list for rapid, blitz list for blitz), joined on the FIDE id. The game month is the game date, or the month of the file if the game has no date. The FIDE tables have one row per FIDE id and month (duplicates dropped, first kept).
* Where PGN and FIDE ratings both exist and differ by more than 100 points, the game is flagged (`*_elo_mismatch`), not changed.
* `analysis_ok` = both ratings known.

In [16]:
fr = read_columns(FIDE_RATINGS, ["fideid", "period", "standard_rating", "rapid_rating", "blitz_rating"])
fr["fideid"], fr["period"] = fr["fideid"].astype(str), fr["period"].astype(str)
n_fr = len(fr)
fr = fr.drop_duplicates(["fideid", "period"]).reset_index(drop=True)
FR_KEY = pd.MultiIndex.from_arrays([fr["fideid"], fr["period"]])
FR_VAL = {k: fr[f"{k}_rating"].to_numpy(dtype="float64", na_value=np.nan) for k in ("standard", "rapid", "blitz")}
def fide_rows(ids, periods):
    """Row of fr for every (FIDE id, month); -1 where the FIDE table has no such row."""
    return FR_KEY.get_indexer(pd.MultiIndex.from_arrays([ids.astype(object).astype(str), periods.astype(str)]))
def fide_value(rows, lst):
    return np.where(rows >= 0, FR_VAL[lst][np.maximum(rows, 0)], np.nan)

period = g["game_date"].dt.strftime("%Y-%m").fillna(g["source_month"]).astype(str)
g["period"] = period
g["date_outside_fide_range"] = (period < FIDE_FIRST) | (period > FIDE_LAST)
valid = lambda x: x.where((x >= 1000) & (x <= 2900))
for side, raw in (("white", we_raw), ("black", be_raw)):
    rows = fide_rows(g[f"{side}_fide_id"], period)
    cls = g["tc_class"]
    fide = pd.Series(np.select([cls == "classical", cls == "rapid", cls == "blitz"],
                               [fide_value(rows, "standard"), fide_value(rows, "rapid"), fide_value(rows, "blitz")], np.nan), index=g.index)
    pgn = valid(raw)
    final = pgn.fillna(fide)
    g[f"{side}_elo_invalid"] = (raw.notna() & pgn.isna()).astype(bool)
    g[f"{side}_elo_fide"] = fide
    g[f"{side}_elo_final"] = final.astype("float64")
    g[f"{side}_elo_source"] = np.select([pgn.notna(), fide.notna()], ["pgn", "fide"], "missing")
    g[f"{side}_elo_mismatch"] = ((pgn - fide).abs() > 100).fillna(False).astype(bool)
g["analysis_ok"] = g["white_elo_final"].notna() & g["black_elo_final"].notna()
del fr, FR_KEY

k = g[kept]
elo = pd.DataFrame({s: k[f"{s}_elo_source"].value_counts() for s in ("white", "black")}).fillna(0).astype(int)
display(elo.rename(index={"pgn": "valid rating in the PGN", "fide": "filled from the FIDE list", "missing": "still missing"}))
inv = int(k.white_elo_invalid.sum() + k.black_elo_invalid.sum()); mm = int(k.white_elo_mismatch.sum() + k.black_elo_mismatch.sum())
filled = int((k.white_elo_source == "fide").sum() + (k.black_elo_source == "fide").sum())
say(f"FIDE table: {n_fr:,} rows, {n_fr - len(FR_VAL['standard']):,} duplicate (id, month) rows dropped. "
    f"**{inv:,}** written ratings were outside 1,000–2,900 and set to missing; **{filled:,}** player ratings were filled from the FIDE list; "
    f"for **{mm:,}** player entries the PGN rating differs from the FIDE list by more than 100 points (flagged, kept). "
    f"Games with both ratings: **{int(k.analysis_ok.sum()):,}** of {len(k):,} ({100 * k.analysis_ok.mean():.1f} %).")
log_step("2 fixes", "PGN rating outside 1000-2900 set to missing (player entries)", "fix", inv, kept.sum())
log_step("2 fixes", "rating filled from the FIDE list of the month and time control (player entries)", "fix", filled, kept.sum())
log_step("2 fixes", "PGN rating differs from FIDE by >100 (player entries, flagged)", "flag", mm, kept.sum())
del k

,white,black
valid rating in the PGN,932836,931901
still missing,194315,195220
filled from the FIDE list,2381,2411


FIDE table: 4,994,547 rows, 0 duplicate (id, month) rows dropped. **35,178** written ratings were outside 1,000–2,900 and set to missing; **4,792** player ratings were filled from the FIDE list; for **21,064** player entries the PGN rating differs from the FIDE list by more than 100 points (flagged, kept). Games with both ratings: **891,337** of 1,129,532 (78.9 %).

### 4.8 Player names

The PGN gives names in free order and spelling; the FIDE list writes `Surname, Firstname`. Matching on names would join wrongly or not at all, so players are identified by their FIDE id, which both sources carry. The cell measures how much that matters.

In [17]:
nm = read_columns(ALL_GAMES, ["white", "black", "white_fide_id", "black_fide_id"])
ids = pd.concat([nm[["white_fide_id", "white"]].set_axis(["id", "name"], axis=1), nm[["black_fide_id", "black"]].set_axis(["id", "name"], axis=1)], ignore_index=True)
del nm
with_id = ids.id.notna()
per_id = ids[with_id].groupby("id")["name"].nunique()
many = per_id[per_id > 1]
say(f"**{100 * with_id.mean():.1f} %** of player entries carry a FIDE id. **{100 * ids.name.astype(str).str.contains(',').mean():.0f} %** of names are written `Surname, Firstname`, the rest in other orders. "
    f"**{len(many):,} of {len(per_id):,} FIDE ids ({100 * len(many) / max(len(per_id), 1):.1f} %) appear under more than one spelling**; a name-based join would treat each spelling as a different person. "
    f"Placeholder ids such as `0`, `-1` or `-` are shared by many different players; they never match a FIDE list row, so they cannot fill a rating.")
ex = ids[with_id & ids.id.isin(many.index)].drop_duplicates()
ex = ex[ex.id.astype(str).str.fullmatch(r"\d{5,}")].groupby("id")["name"].apply(lambda s: " | ".join(s.astype(str).head(3))).head(5)
display(ex.rename("spellings of one player").to_frame())
del ids

**68.3 %** of player entries carry a FIDE id. **81 %** of names are written `Surname, Firstname`, the rest in other orders. **3,348 of 94,888 FIDE ids (3.5 %) appear under more than one spelling**; a name-based join would treat each spelling as a different person. Placeholder ids such as `0`, `-1` or `-` are shared by many different players; they never match a FIDE list row, so they cannot fill a rating.

,spellings of one player
id,
10000569,"Nsubuga Haruna (UGA) | Nsubuga, Haruna"
100013,"Campora, Daniel H. | Daniel H. Campora"
1000330,"Grooten, Herman | Herman Grooten"
1001183,"Becx, Cesar | César Becx"
10014373,"Naisanga, Sheba Valentine | Naisanga Sheba Valentine"


### 4.9 Result of the cleaning rules

The games that pass rules 1-10 are written to `interim/clean_games.parquet` with all their raw columns and the repaired fields; the removed games go to `interim/excluded_games.parquet` with their reason.

In [18]:
NEW_COLS = ["base_hat", "inc_hat", "score_white", "tc_method", "tc_base_s", "tc_inc_s", "tc_class", "tc_source", "tc_borderline", "period",
            "date_outside_fide_range"] + [f"{s}_elo_{x}" for s in ("white", "black") for x in ("invalid", "fide", "final", "source", "mismatch")] + ["analysis_ok"]
kept = g["reason"].isna().to_numpy()
write_rows(ALL_GAMES, CLEAN_GAMES, kept, g.loc[kept, NEW_COLS])
write_parquet(g.loc[~kept, ["game_url", "reason", "broadcast_name", "source_month"]], EXCLUDED)
save_log("4_rules")
lg = pd.read_csv(os.path.join(INTERIM, "_log_4_rules.csv"))
display(lg[lg.action.isin(["start", "remove"])][["step", "rows_affected", "rows_left"]].rename(columns={"rows_affected": "games removed", "rows_left": "games left"}).set_index("step"))
say(f"**{int(kept.sum()):,}** of {N0:,} games pass the cleaning rules; **{int(g.loc[kept, 'analysis_ok'].sum()):,}** of them have both ratings.")
del g, tcs, sb, si, clk0, string_ok, agrees, use_string, use_clock, snap_b, snap_i, base_c, inc_c, cs, cc, name, term, var, r, rkind

,games removed,games left
step,,
all games parsed (notebook 01),1263916,1263916
1 unfinished result (*),90666,1173250
2 forfeit result code,29,1173221
3 invalid result string,27,1173194
4 non-standard variant / Chess960 / start position,13382,1159812
5 no moves / unplayed (incl. forfeits),21465,1138347
6 engine-level Elo (2900-4000),8383,1129964
7 adjudicated game,1,1129963
8 engine/adjudicated event (>=50% of its games flagged),116,1129847


**1,129,532** of 1,263,916 games pass the cleaning rules; **891,337** of them have both ratings.

## 5. Why ratings are missing, and why PGN and FIDE sometimes disagree

For every player entry with no usable rating but a FIDE id, the cell looks for the reason; then it compares the PGN ratings with the FIDE list where both exist.

In [19]:
k = read_columns(CLEAN_GAMES, ["white_fide_id", "black_fide_id", "period", "tc_class", "date_outside_fide_range", "analysis_ok"] +
                 [f"{s}_elo_{x}" for s in ("white", "black") for x in ("source", "final", "fide", "mismatch")])
fr = read_columns(FIDE_RATINGS, ["fideid", "period", "standard_rating", "rapid_rating", "blitz_rating"])
fr["fideid"], fr["period"] = fr["fideid"].astype(str), fr["period"].astype(str)
fr = fr.drop_duplicates(["fideid", "period"]).reset_index(drop=True)
FR_KEY = pd.MultiIndex.from_arrays([fr["fideid"], fr["period"]])
FR = fr[["standard_rating", "rapid_rating", "blitz_rating"]].to_numpy(dtype="float64", na_value=np.nan)
known_ids = set(pq.read_table(FIDE_PLAYERS, columns=["fideid"]).column(0).to_pylist())
rows_out, which = [], []
for side in ("white", "black"):
    ids = k[f"{side}_fide_id"].astype(object).astype(str)
    rows = FR_KEY.get_indexer(pd.MultiIndex.from_arrays([ids, k["period"].astype(str)]))
    got = np.where((rows >= 0)[:, None], FR[np.maximum(rows, 0)], np.nan)
    any_rating = ~np.isnan(got).all(axis=1)
    miss = (k[f"{side}_elo_source"] == "missing") & k[f"{side}_fide_id"].notna()
    in_lists = ids.isin(list(known_ids))
    both = (k[f"{side}_elo_source"] == "pgn") & k[f"{side}_elo_fide"].notna()
    d = (k[f"{side}_elo_final"] - k[f"{side}_elo_fide"]).abs()
    big = (both & (d > 100)).to_numpy()
    pg = k.loc[big, f"{side}_elo_final"].to_numpy(float)
    near = (np.where(np.isnan(got[big]), 1e9, np.abs(got[big] - pg[:, None])).min(axis=1) <= 25) if big.any() else np.array([], bool)
    rows_out.append({"side": side, "rating missing, FIDE id present": int(miss.sum()),
        "· id in no FIDE list (unrated, placeholder id)": int((miss & ~in_lists).sum()),
        "· id known, game month outside the FIDE months": int((miss & in_lists & k.date_outside_fide_range).sum()),
        "· id known, no rating in that month's lists": int((miss & in_lists & ~k.date_outside_fide_range & ~any_rating).sum()),
        "· rating exists that month, but not in the list of the game's time control": int((miss & any_rating).sum()),
        "PGN and FIDE both present": int(both.sum()), "· differ by more than 100": int(big.sum()),
        "· · PGN equals another FIDE list within 25 (list confusion)": int(near.sum()),
        "· · matches no list (typing error or stale rating)": int((~near).sum())})
    for j, lst in enumerate(("standard", "rapid", "blitz")):
        which.append(pd.DataFrame({"tc_class": k.loc[big, "tc_class"].to_numpy(), "list": lst, "hit": np.abs(got[big, j] - pg) <= 25}))
display(pd.DataFrame(rows_out).set_index("side").T)
w = pd.concat(which)
if len(w):
    say("Share of the large PGN–FIDE differences in which the PGN rating equals (within 25 points) the rating of each FIDE list, by the game's time control:")
    display(w.groupby(["tc_class", "list"])["hit"].mean().unstack().round(3))
say("**Reading.** The FIDE lookup can fill only few ratings: a player without a PGN rating is usually not on that month's list at all (unrated, below the floor, or a placeholder id). "
    "Most large PGN–FIDE differences occur in rapid and blitz games whose PGN carries the player's **standard** rating (the organiser published the standard rating): "
    "list confusion, not an error in the game. These games are kept and flagged; the analysis is re-run on verified ratings only in notebook 03.")
del k, fr, FR_KEY, FR

side,white,black
"rating missing, FIDE id present",49277,50006
"· id in no FIDE list (unrated, placeholder id)",13808,14004
"· id known, game month outside the FIDE months",0,0
"· id known, no rating in that month's lists",20051,20387
"· rating exists that month, but not in the list of the game's time control",15418,15615
PGN and FIDE both present,720894,720221
· differ by more than 100,10516,10548
· · PGN equals another FIDE list within 25 (list confusion),9036,9082
· · matches no list (typing error or stale rating),1480,1466


Share of the large PGN–FIDE differences in which the PGN rating equals (within 25 points) the rating of each FIDE list, by the game's time control:

list,blitz,rapid,standard
tc_class,,,
blitz,0.000,0.246,0.819
classical,0.195,0.323,0.000
rapid,0.103,0.000,0.827


**Reading.** The FIDE lookup can fill only few ratings: a player without a PGN rating is usually not on that month's list at all (unrated, below the floor, or a placeholder id). Most large PGN–FIDE differences occur in rapid and blitz games whose PGN carries the player's **standard** rating (the organiser published the standard rating): list confusion, not an error in the game. These games are kept and flagged; the analysis is re-run on verified ratings only in notebook 03.

## 6. Missing values

### 6.1 How many, per column

Missing values in every column of the table after the cleaning rules (read from the Parquet metadata). Most tag columns are empty by design (they exist only for some events); the fields the analysis needs are the result (no longer missing: games without a result were removed), the time control and the two ratings.

In [20]:
pf = pq.ParquetFile(CLEAN_GAMES); n_rows = pf.metadata.num_rows
nulls = {f.name: 0 for f in pf.schema_arrow}
for i in range(pf.metadata.num_row_groups):
    for j, f in enumerate(pf.schema_arrow):
        st = pf.metadata.row_group(i).column(j).statistics
        nulls[f.name] += st.null_count if st is not None and st.null_count is not None else 0
miss = pd.DataFrame({"missing": pd.Series(nulls)}).assign(share_pct=lambda d: (100 * d.missing / n_rows).round(1))
tcu = int((pq.read_table(CLEAN_GAMES, columns=["tc_class"]).column(0).to_pandas() == "unknown").sum())
miss.loc["tc_class (value 'unknown')"] = [tcu, round(100 * tcu / n_rows, 1)]
say(f"Table after the cleaning rules: {n_rows:,} games × {len(nulls)} columns; {int((miss.missing == 0).sum())} columns have no missing value.")
display(miss.sort_values("missing", ascending=False))

Table after the cleaning rules: 1,129,532 games × 74 columns; 32 columns have no missing value.

,missing,share_pct
set_up,1129532,100.0
fen,1129532,100.0
white_clock,1127705,99.8
black_clock,1127705,99.8
white_country,1124774,99.6
black_country,1124754,99.6
game_id,1122114,99.3
termination,1097459,97.2
chapter_url,1069530,94.7
annotator,1060677,93.9


### 6.2 Pattern: by year, time control and event

The Elo is *missing* for a game if either player has no usable rating after the FIDE lookup. The games dated before 2019 (date artefacts, removed in section 7) are left out here.

In [21]:
cm = read_columns(CLEAN_GAMES, ["period", "analysis_ok", "tc_class", "tc_source", "broadcast_name", "n_plies", "score_white", "white_elo_source", "black_elo_source",
                                "white_fide_id", "black_fide_id"])
cm["year"] = pd.to_numeric(cm.period.astype(str).str[:4], errors="coerce")
cm = cm[cm.year >= 2019].reset_index(drop=True)
cm["elo_missing"] = ~cm.analysis_ok
cm["tc_missing"] = cm.tc_class.eq("unknown")
by_year = cm.groupby("year").agg(games=("elo_missing", "size"), elo_missing_pct=("elo_missing", "mean"), tc_missing_pct=("tc_missing", "mean"))
by_tc = cm.groupby("tc_class").agg(games=("elo_missing", "size"), elo_missing_pct=("elo_missing", "mean"))
display(pd.concat([by_year.assign(elo_missing_pct=lambda d: (100 * d.elo_missing_pct).round(1), tc_missing_pct=lambda d: (100 * d.tc_missing_pct).round(1))
                   .rename(index=lambda y: f"year {int(y)}"),
                   by_tc.assign(elo_missing_pct=lambda d: (100 * d.elo_missing_pct).round(1)).rename(index=lambda c: f"time control {c}")]))
ev = cm.groupby("broadcast_name").agg(n=("elo_missing", "size"), share=("elo_missing", "mean"))
ev = ev[ev.n >= 100]
one = int((cm.white_elo_source.eq("missing") ^ cm.black_elo_source.eq("missing")).sum())
both_m = int((cm.white_elo_source.eq("missing") & cm.black_elo_source.eq("missing")).sum())
worst50 = cm.broadcast_name.isin(ev.sort_values("share", ascending=False).head(50).index) & cm.elo_missing
say(f"Overall **{100 * cm.elo_missing.mean():.1f} %** of games ({int(cm.elo_missing.sum()):,}) have no usable rating for at least one player, and **{100 * cm.tc_missing.mean():.1f} %** "
    f"have no usable time control. Of **{len(ev):,}** events with at least 100 games, **{int((ev.share < 0.05).sum()):,}** have almost no missing rating and "
    f"**{int((ev.share > 0.9).sum()):,}** have more than 90 % missing: organisers publish ratings for all games or for none. "
    f"Both ratings are missing in **{both_m:,}** games and exactly one in {one:,}; if the two sides were missing independently, both would be missing in only about "
    f"{(cm.elo_missing.mean() ** 2) * len(cm):,.0f} games. The 50 worst events hold {100 * worst50.sum() / max(cm.elo_missing.sum(), 1):.1f} % of the missing games, so the problem is spread over many events.")

,games,elo_missing_pct,tc_missing_pct
year 2019,256,0.0,0.0
year 2020,6656,15.8,3.2
year 2021,75007,35.3,3.5
year 2022,89269,34.6,1.9
year 2023,139770,41.4,3.3
year 2024,241392,16.3,3.9
year 2025,306102,14.5,2.1
year 2026,271053,14.0,1.7
time control blitz,134873,20.0,NaN
time control classical,767886,19.6,NaN


Overall **21.1 %** of games (238,168) have no usable rating for at least one player, and **2.6 %** have no usable time control. Of **3,177** events with at least 100 games, **1,798** have almost no missing rating and **352** have more than 90 % missing: organisers publish ratings for all games or for none. Both ratings are missing in **151,313** games and exactly one in 86,855; if the two sides were missing independently, both would be missing in only about 50,220 games. The 50 worst events hold 3.9 % of the missing games, so the problem is spread over many events.

### 6.3 Mechanism: MCAR, MAR or MNAR?

- MCAR (missing completely at random): missingness unrelated to anything. The missing games would then look like the others and the missing rate would be similar everywhere.
- MAR (missing at random given observed variables): missingness depends on recorded variables (event, year, time control), not on the missing rating itself.
- MNAR (missing not at random): missingness depends on the unobserved rating itself, for example because unrated players are weaker.

Three tests: a chi-square test of independence of "rating missing" against year and against event, with Cramér's V as effect size (0 = none, 1 = complete); permutation tests (1,000 shuffles) of whether games with a missing rating differ in length, draw rate or White's score; and a logistic regression of "rating missing" on the observed variables, whose AUC says how predictable missingness is (0.5 = not at all).

In [22]:
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
rows = []
big = cm.broadcast_name.map(cm.broadcast_name.value_counts()) >= 100
for resp in ("elo_missing", "tc_missing"):
    for factor, d, key in (("year", cm, cm.year), ("event (broadcast)", cm[big], cm.broadcast_name[big])):
        tab = pd.crosstab(key, d[resp])
        if min(tab.shape) < 2:
            rows.append(dict(missing=resp.replace("_missing", ""), against=factor, games=int(tab.values.sum()), categories=tab.shape[0])); continue
        chi2, p, dof, _ = chi2_contingency(tab)
        rows.append(dict(missing=resp.replace("_missing", ""), against=factor, games=int(tab.values.sum()), categories=tab.shape[0], chi2=round(chi2), dof=dof,
                         p_value="< 0.001" if p < 0.001 else f"{p:.3f}", cramers_v=round(float(np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1)))), 2)))
chi = pd.DataFrame(rows); display(chi)

rng = np.random.default_rng(SEED)
def perm(x, m, reps=1000):
    x = x.to_numpy(float); m = m.to_numpy(bool); obs = x[m].mean() - x[~m].mean(); cnt = 0
    for _ in range(reps):
        pm = rng.permutation(m); cnt += abs(x[pm].mean() - x[~pm].mean()) >= abs(obs)
    return obs, (cnt + 1) / (reps + 1)
cm["is_draw"] = (cm.score_white == 0.5).astype(float)
pt = []
for c, label in (("n_plies", "game length (half-moves)"), ("is_draw", "draw rate"), ("score_white", "White's score")):
    s = cm.sample(300_000, random_state=1) if len(cm) > 300_000 else cm
    d, p = perm(s[c], s["elo_missing"]); pt.append(("permutation", label, round(d, 4), round(p, 4)))
perm_tab = pd.DataFrame(pt, columns=["test", "variable", "difference (missing minus observed)", "p-value"]); display(perm_tab.drop(columns="test"))

s = cm.sample(min(300_000, len(cm)), random_state=1)
X = pd.get_dummies(s[["year", "tc_class"]].astype(str), drop_first=True).astype(float)
X["plies_z"] = (s.n_plies - s.n_plies.mean()) / s.n_plies.std()
X["is_draw"] = s["is_draw"]; X["has_clock"] = s["tc_source"].isin(["clock_raw", "clock_snapped", "clock_override"]).astype(float)
y = s["elo_missing"].astype(int)
if y.nunique() == 2:
    lr = LogisticRegression(max_iter=1000).fit(X, y)
    auc = roc_auc_score(y, lr.predict_proba(X)[:, 1])
    odds = pd.Series(np.exp(lr.coef_[0]), index=X.columns).round(2).sort_values(ascending=False)
    say(f"Logistic model of 'a rating is missing' on {len(s):,} games: **AUC = {auc:.3f}**. Odds ratios (above 1 = more likely missing): " + ", ".join(f"{k} {v}" for k, v in odds.items()))
else:
    auc = np.nan; say("Logistic model skipped: every game in the data has the same missing status.")
tests = pd.concat([chi.assign(test="chi-square"), perm_tab, pd.DataFrame([dict(test="logistic AUC", variable="year, time control, length, draw, clock", **{"p-value": None}, auc=auc)])], ignore_index=True)
tests.to_csv(os.path.join(REPORTS, "missingness_tests.csv"), index=False)
ce = chi[(chi.missing == "elo") & (chi.against == "event (broadcast)")]; cy = chi[(chi.missing == "elo") & (chi.against == "year")]
v_ev = ce.cramers_v.iloc[0] if "cramers_v" in ce and ce.cramers_v.notna().any() else np.nan
v_yr = cy.cramers_v.iloc[0] if "cramers_v" in cy and cy.cramers_v.notna().any() else np.nan
say(f"**Reading.** A missing rating depends strongly on the event (Cramér's V = **{v_ev}**) and moderately on the year (V = **{v_yr}**); games with a missing rating are also shorter and drawn less often "
    f"(permutation p-values at the smallest value 1,000 shuffles can give), and the logistic model predicts missingness well above chance (AUC {auc:.2f}). **MCAR is rejected.** "
    "The pattern (all-or-nothing by event, whole years of weaker reporting in 2021–2023) is explained by recorded variables, i.e. it fits **MAR**. "
    "**MNAR cannot be tested from the data**: unrated players are mostly juniors and amateurs, who are weaker on average, so a dependence on the unobserved rating is plausible; it is stated as a limitation.")

,missing,against,games,categories,chi2,dof,p_value,cramers_v
0,elo,year,1129505,8,72919,7,< 0.001,0.25
1,elo,event (broadcast),787639,3177,523402,3176,< 0.001,0.82
2,tc,year,1129505,8,3420,7,< 0.001,0.06
3,tc,event (broadcast),787639,3177,393692,3176,< 0.001,0.71


,variable,difference (missing minus observed),p-value
0,game length (half-moves),-4.8926,0.001
1,draw rate,-0.0814,0.001
2,White's score,-0.0139,0.001


Logistic model of 'a rating is missing' on 300,000 games: **AUC = 0.691**. Odds ratios (above 1 = more likely missing): year_2023 2.05, year_2022 1.55, tc_class_rapid 1.54, has_clock 1.43, year_2021 1.43, tc_class_unknown 1.1, tc_class_classical 0.93, plies_z 0.84, is_draw 0.59, year_2024 0.54, year_2025 0.53, year_2020 0.5, year_2026 0.5

**Reading.** A missing rating depends strongly on the event (Cramér's V = **0.82**) and moderately on the year (V = **0.25**); games with a missing rating are also shorter and drawn less often (permutation p-values at the smallest value 1,000 shuffles can give), and the logistic model predicts missingness well above chance (AUC 0.69). **MCAR is rejected.** The pattern (all-or-nothing by event, whole years of weaker reporting in 2021–2023) is explained by recorded variables, i.e. it fits **MAR**. **MNAR cannot be tested from the data**: unrated players are mostly juniors and amateurs, who are weaker on average, so a dependence on the unobserved rating is plausible; it is stated as a limitation.

### 6.4 Imputation or deletion

| Field | Choice | Reason |
|---|---|---|
| Rating | Filled from the FIDE list of the game's month and time control where such a rating exists (section 4.7); otherwise the game is deleted | A statistical imputation (mean, regression, nearest neighbour) would insert a value for the variable under study: the rating gap *is* the explanatory variable, so imputed ratings would pull gaps towards the mean and bias the critical gap. The missingness is event-level (MAR), so deleting the games does not distort comparisons within events, but the results describe rated players only. |
| Time control | Repaired from the string, the clock or the event name (section 4.6); the rest is deleted | The class is the grouping variable of the question; a guessed class would blur the comparison between time controls. |
| Result | Deleted if unfinished, forfeited or invalid (4.1) | There is no score to analyse. |
| Game date | Missing date replaced by the month of the file (`period`) | Only the month is needed (to find the FIDE list); the file month is at most a few days off. |
| Other tags | Kept as they are | Not used in the analysis. |

In [23]:
say(f"Deleting the games without both ratings keeps **{int((~cm.elo_missing).sum()):,}** of {len(cm):,} games ({100 * (~cm.elo_missing).mean():.1f} %). "
    "The deletion itself is done in section 7, with the other analysis filters.")
del cm, s, X

Deleting the games without both ratings keeps **891,337** of 1,129,505 games (78.9 %). The deletion itself is done in section 7, with the other analysis filters.

## 7. Analysis filters and derived columns

**Repair of glitchy clock estimates.** When a game has no usable time-control string, its time control comes from the clocks. A glitch in the clock feed can give an impossible increment (more than 60 s), which always pushes the game into *classical*. All games of one round share a time control, so such a game takes the dominant time control of the verified games of the same round (`broadcast_url`); failing that, of the same tournament (`broadcast_name`), if one time control covers at least 80 % of its verified games; failing that the time control is `unknown`.

Filters (in this order): game dated 2019 or later (earlier dates are artefacts such as `2010.01.01`); both ratings known; time control known; at least 10 half-moves (shorter "games" are walkovers or pre-arranged results recorded as games; see section 8).

**Derived columns:** rating gap, average rating, who is the underdog (lower-rated player) and its colour and score, rating band, 50-point gap bin, the FIDE flags `elo_mismatch_any` and `verified`.

In [24]:
g = read_columns(CLEAN_GAMES, ["tc_source", "tc_base_s", "tc_inc_s", "tc_class", "tc_borderline", "broadcast_url", "broadcast_name", "period", "game_date",
                               "analysis_ok", "n_plies", "score_white", "white_elo_final", "black_elo_final", "white_elo_fide", "black_elo_fide",
                               "white_elo_mismatch", "black_elo_mismatch"])
g["src_row"] = np.arange(len(g))
n_start = len(g)
log_step("3 analysis filters", "start (games that pass the cleaning rules)", "start", n_start, n_start)
bad = (g["tc_source"] == "clock_raw") & (g["tc_inc_s"] > 60)
good = g[g["tc_source"].isin(["string_verified_by_clock", "clock_snapped"]) & g["tc_base_s"].notna()]
def dominant(key):
    c = good.groupby([key, "tc_base_s", "tc_inc_s"]).size().rename("n").reset_index()
    c["share"] = c["n"] / c.groupby(key)["n"].transform("sum")
    return c.sort_values("n", ascending=False).drop_duplicates(key).query("share >= 0.8").set_index(key)[["tc_base_s", "tc_inc_s"]]
fix = pd.DataFrame(index=g.index, columns=["b", "i", "how"], dtype=object)
for key, how in (("broadcast_url", "round_consensus"), ("broadcast_name", "tournament_consensus")):
    dom = dominant(key)
    m = bad & fix["b"].isna() & g[key].isin(dom.index)
    fix.loc[m, "b"] = g.loc[m, key].map(dom["tc_base_s"]); fix.loc[m, "i"] = g.loc[m, key].map(dom["tc_inc_s"]); fix.loc[m, "how"] = how
done = fix["b"].notna()
tot = fix.loc[done, "b"].astype(float) + 60 * fix.loc[done, "i"].astype(float)
old_class = g.loc[done, "tc_class"].copy()
g.loc[done, "tc_base_s"] = fix.loc[done, "b"].astype(float); g.loc[done, "tc_inc_s"] = fix.loc[done, "i"].astype(float)
g.loc[done, "tc_class"] = np.select([tot <= 600, tot < 3600], ["blitz", "rapid"], "classical")
g.loc[done, "tc_source"] = fix.loc[done, "how"]
g.loc[done, "tc_borderline"] = (np.minimum((tot - 600).abs(), (tot - 3600).abs()) <= 180)
unfixed = bad & ~done
g.loc[unfixed, "tc_class"] = "unknown"; g.loc[unfixed, "tc_source"] = "clock_glitch_unresolved"
changed = int((old_class != g.loc[done, "tc_class"]).sum())
log_step("2 fixes", "glitchy clock estimate (increment > 60 s) repaired from round/tournament consensus", "fix", done.sum(), n_start)
log_step("2 fixes", "glitchy clock estimate that could not be repaired -> time control unknown", "fix", unfixed.sum(), n_start)
say(f"Glitchy clock estimates: **{int(bad.sum()):,}** games; repaired from the round or tournament **{int(done.sum()):,}** "
    f"({int((fix.how == 'round_consensus').sum()):,} round, {int((fix.how == 'tournament_consensus').sum()):,} tournament), of which {changed:,} changed class; "
    f"**{int(unfixed.sum()):,}** set to unknown.")

g["year"] = pd.to_numeric(g["period"].astype(str).str[:4], errors="coerce")
n_nodate = int(g["game_date"].isna().sum())
log_step("2 fixes", "no game date: month of the file used as period", "fix", n_nodate, n_start)
ok = pd.Series(True, index=g.index)
reason = pd.Series(None, index=g.index, dtype="object")
for mask, label in ((g["year"] >= 2019, "11 drop games dated before 2019 (date artefacts)"), (g["analysis_ok"].astype(bool), "12 both Elos known"),
                    (g["tc_class"] != "unknown", "13 time control known (incl. unresolved glitchy clocks)"),
                    (g["n_plies"] >= 10, "14 at least 10 plies (shorter results are walkovers/forfeits recorded as games)")):
    new = ok & ~mask.fillna(False)
    reason[new] = label; ok &= ~new
    log_step("3 analysis filters", label, "remove", new.sum(), ok.sum())
write_parquet(pd.DataFrame({"game_url": read_columns(CLEAN_GAMES, ["game_url"])["game_url"][~ok.to_numpy()], "reason": reason[~ok]}),
              os.path.join(INTERIM, "excluded_analysis_filters.parquet"))
a = g[ok].reset_index(drop=True)

Glitchy clock estimates: **13,790** games; repaired from the round or tournament **12,828** (11,488 round, 1,340 tournament), of which 878 changed class; **962** set to unknown.

In [25]:
we, be = a["white_elo_final"].astype(float), a["black_elo_final"].astype(float)
a["white_elo_final"], a["black_elo_final"] = we.round().astype("int64"), be.round().astype("int64")
a["gap"] = (we - be).abs()
a["avg_elo"] = (we + be) / 2
a["has_underdog"] = a["gap"] > 0
a["underdog_is_white"] = we < be
a["fav_is_white"] = we > be
sw = a["score_white"].astype(float)
a["underdog_score"] = np.where(a["underdog_is_white"], sw, 1 - sw)              # 1 win, 0.5 draw, 0 loss
a["underdog_color"] = np.where(a["underdog_is_white"], "white", "black")
a["band"] = pd.cut(a["avg_elo"], [0, 2200, 2500, 5000], right=False, labels=["<2200", "2200-2499", "2500+"]).astype(str)
a["gap_bin"] = (np.minimum(a["gap"], 600) // 50 * 50).astype(int)                 # lower edge; 600 = "600+"
a["elo_mismatch_any"] = a["white_elo_mismatch"].astype(bool) | a["black_elo_mismatch"].astype(bool)
okw = a["white_elo_fide"].notna() & ((a["white_elo_final"] - a["white_elo_fide"]).abs() <= 100)
okb = a["black_elo_fide"].notna() & ((a["black_elo_final"] - a["black_elo_fide"]).abs() <= 100)
a["verified"] = okw & okb
a["year"] = a["year"].astype("int64")
a["tc_borderline"] = a["tc_borderline"].astype(bool)
DER_COLS = ["src_row", "tc_base_s", "tc_inc_s", "tc_class", "tc_source", "tc_borderline", "year", "white_elo_final", "black_elo_final", "gap", "avg_elo",
            "has_underdog", "underdog_is_white", "fav_is_white", "underdog_score", "underdog_color", "band", "gap_bin", "elo_mismatch_any", "verified",
            "n_plies", "score_white"]
write_parquet(a[DER_COLS], DERIVED)
save_log("7_filters")
lg = pd.read_csv(os.path.join(INTERIM, "_log_7_filters.csv")); lg = lg[lg.action != "fix"]
display(lg[["step", "rows_affected", "rows_left"]].rename(columns={"rows_affected": "games removed", "rows_left": "games left"}).set_index("step"))
say(f"Analysis set: **{len(a):,}** games, **{int(a.has_underdog.sum()):,}** with an underdog (rating gap > 0) and {int((~a.has_underdog).sum()):,} exact ties. "
    f"By time control: " + ", ".join(f"{c} {int((a.tc_class == c).sum()):,}" for c in TC) +
    f". Both ratings confirmed by the FIDE list (`verified`): {int(a.verified.sum()):,} games ({100 * a.verified.mean():.1f} %).")
del g, good, fix

,games removed,games left
step,,
start (games that pass the cleaning rules),1129532,1129532
11 drop games dated before 2019 (date artefacts),27,1129505
12 both Elos known,238168,891337
13 time control known (incl. unresolved glitchy clocks),23938,867399
14 at least 10 plies (shorter results are walkovers/forfeits recorded as games),3428,863971


Analysis set: **863,971** games, **862,599** with an underdog (rating gap > 0) and 1,372 exact ties. By time control: classical 613,735, rapid 142,763, blitz 107,473. Both ratings confirmed by the FIDE list (`verified`): 657,365 games (76.1 %).

## 8. Outliers: detection, diagnosis, decision

| Variable | Method | Rule |
|---|---|---|
| Game length (half-moves), per time control | IQR fence and MAD robust z-score | beyond Q1 − 1.5 IQR / Q3 + 1.5 IQR; robust z = \|x − median\| / (1.4826 · MAD) > 3.5 |
| Ratings | plausible range | below 1,000 or above 2,900 is not a human rating (section 4.4 and 4.7) |
| (rating gap, average rating, length, underdog's score), per time control | Isolation Forest, the 1 % most isolated games | multivariate: finds unusual *combinations* |

An outlier is not removed because it is extreme: each class is first diagnosed as an error or a real value.

In [26]:
a = pd.read_parquet(DERIVED)
rows = []
a["len_outlier_iqr"] = False; a["len_outlier_mad"] = False
for c in TC:
    sel = a.tc_class == c
    x = a.loc[sel, "n_plies"].astype(float)
    if not len(x): continue
    q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1; med = x.median(); mad = 1.4826 * (x - med).abs().median()
    a.loc[sel, "len_outlier_iqr"] = (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)
    a.loc[sel, "len_outlier_mad"] = (x - med).abs() / max(mad, 1e-9) > 3.5
    rows.append(dict(time_control=c, games=len(x), median=med, IQR_fence_low=q1 - 1.5 * iqr, IQR_fence_high=q3 + 1.5 * iqr,
                     IQR_outliers=int(a.loc[sel, "len_outlier_iqr"].sum()), MAD_outliers=int(a.loc[sel, "len_outlier_mad"].sum()),
                     longest=int(x.max()), under_20_half_moves=int((x < 20).sum())))
len_tab = pd.DataFrame(rows).set_index("time_control").round(1); display(len_tab)
say(f"All length outliers lie **above** the upper fence (the lower fence is negative): long games of {len_tab.IQR_fence_high.min():.0f}+ half-moves, at most {len_tab.longest.max()} half-moves.")

,games,median,IQR_fence_low,IQR_fence_high,IQR_outliers,MAD_outliers,longest,under_20_half_moves
time_control,,,,,,,,
classical,613735,81.0,-4.0,172.0,12228,6755,599,4561
rapid,142763,85.0,-2.5,177.5,3003,1452,453,768
blitz,107473,83.0,-6.0,178.0,2255,1173,496,610


All length outliers lie **above** the upper fence (the lower fence is negative): long games of 172+ half-moves, at most 599 half-moves.

In [27]:
from sklearn.ensemble import IsolationForest
FEAT = ["gap", "avg_elo", "n_plies", "underdog_score"]
a["iforest_outlier"] = False
res = []
for c in TC:
    u = a[a.has_underdog & (a.tc_class == c)]
    if len(u) < 100: continue
    d = u.sample(min(150_000, len(u)), random_state=3)
    model = IsolationForest(contamination=0.01, random_state=0, n_jobs=1).fit(d[FEAT].to_numpy(float))
    flag = model.predict(d[FEAT].to_numpy(float)) == -1
    a.loc[u.index, "iforest_outlier"] = model.predict(u[FEAT].to_numpy(float)) == -1
    big_upset = ((d.gap >= 300) & (d.underdog_score > 0)).to_numpy()
    res.append(dict(time_control=c, games_fitted=len(d), outliers=int(flag.sum()),
                    mean_gap_outliers=round(d.gap.to_numpy()[flag].mean()), mean_gap_others=round(d.gap.to_numpy()[~flag].mean()),
                    underdog_win_pct_outliers=round(100 * (d.underdog_score.to_numpy()[flag] == 1).mean(), 1), underdog_win_pct_others=round(100 * (d.underdog_score.to_numpy()[~flag] == 1).mean(), 1),
                    gap300_underdog_scores_pct_outliers=round(100 * big_upset[flag].mean(), 1), gap300_underdog_scores_pct_others=round(100 * big_upset[~flag].mean(), 1),
                    flagged_in_all_games=int(a.loc[u.index, "iforest_outlier"].sum())))
iso = pd.DataFrame(res).set_index("time_control"); display(iso.T)
a["short_game"] = a["n_plies"] < 20
write_parquet(a[["src_row", "len_outlier_iqr", "len_outlier_mad", "short_game", "iforest_outlier"]], FLAGS)
pd.concat([len_tab.T, iso.T]).to_csv(os.path.join(REPORTS, "outlier_summary.csv"))
for c_, label in (("len_outlier_iqr", "game length beyond the IQR fence (flagged, kept)"), ("len_outlier_mad", "game length with robust z > 3.5 (flagged, kept)"),
                  ("iforest_outlier", "Isolation Forest outlier (flagged, kept)")):
    log_step("4 outliers", label, "flag", a[c_].sum(), len(a))
save_log("8_outliers")
if len(iso):
    say(f"Isolation-Forest outliers have a mean gap of {iso.mean_gap_outliers.min():.0f}–{iso.mean_gap_outliers.max():.0f} points against {iso.mean_gap_others.min():.0f}–{iso.mean_gap_others.max():.0f} for the other games, "
        f"and **{iso.gap300_underdog_scores_pct_outliers.min():.0f}–{iso.gap300_underdog_scores_pct_outliers.max():.0f} %** of them are games in which an underdog at least 300 points weaker scored, "
        f"against {iso.gap300_underdog_scores_pct_others.min():.1f}–{iso.gap300_underdog_scores_pct_others.max():.1f} % of the others: the method finds exactly the large-gap upsets the project is about.")

time_control,classical,rapid,blitz
games_fitted,150000.0,142529.0,107311.0
outliers,1500.0,1426.0,1074.0
mean_gap_outliers,356.0,400.0,321.0
mean_gap_others,178.0,218.0,206.0
underdog_win_pct_outliers,41.3,31.9,38.6
underdog_win_pct_others,16.5,20.4,24.5
gap300_underdog_scores_pct_outliers,27.6,25.7,29.1
gap300_underdog_scores_pct_others,3.1,4.7,4.1
flagged_in_all_games,6027.0,1426.0,1074.0


Isolation-Forest outliers have a mean gap of 321–400 points against 178–218 for the other games, and **26–29 %** of them are games in which an underdog at least 300 points weaker scored, against 3.1–4.7 % of the others: the method finds exactly the large-gap upsets the project is about.

**Diagnosis and decision.** The 200 rarest upsets are also listed with their Lichess links in notebook 03 (`reports/upset_audit_top.csv`), so that they can be opened and checked by hand.

| Outlier class | Diagnosis | Decision |
|---|---|---|
| Games with no moves, or fewer than 10 half-moves | Error: walkovers and forfeits recorded as games | Removed (rule 5, filter 14) |
| Engine-level ratings (2,900-4,000) and events where at least half of the games are engine or adjudicated games | Error: computer players, not people | Removed (rules 6-8) |
| Ratings below 1,000 or above 4,000, placeholder values | Error: not a plausible over-the-board rating | Set to missing, replaced from FIDE where possible (4.7) |
| Very long games (beyond the IQR fence / MAD > 3.5) | Real: long endgames, the rules allow them | Kept, flagged `len_outlier_iqr`, `len_outlier_mad` |
| Short games of 10-19 half-moves | Real in most cases (quick draws by agreement, early blunders), cannot be told apart from pre-arranged results | Kept, flagged `short_game` |
| Isolation-Forest outliers | Real: mostly large-gap upsets; an upset is by definition an unusual combination | Kept, flagged `iforest_outlier` |
| PGN rating differing from FIDE by more than 100 | Uncertain: mostly the standard rating published on a rapid/blitz game (section 5), sometimes a typing error or a stale list | Kept, flagged `elo_mismatch_any`; the analysis is repeated on `verified` games in notebook 03 |

## 9. Before and after cleaning

The cleaning rules act most on two variables: game length (walkovers disappear) and the players' rating (engine-level and implausible ratings disappear). The table compares the raw table with the analysis set.

In [28]:
raw = read_columns(ALL_GAMES, ["n_plies", "white_elo", "black_elo"])
for c in ("white_elo", "black_elo"): raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw_avg = ((raw.white_elo + raw.black_elo) / 2).dropna()
a = pd.read_parquet(DERIVED, columns=["n_plies", "avg_elo", "gap"])
raw_gap = (raw.white_elo - raw.black_elo).abs().dropna()
cmp_ = pd.DataFrame({
    "games": [len(raw), len(a)],
    "median half-moves": [raw.n_plies.median(), a.n_plies.median()],
    "share < 10 half-moves (%)": [100 * (raw.n_plies < 10).mean(), 100 * (a.n_plies < 10).mean()],
    "games with both PGN ratings": [len(raw_avg), len(a)],
    "median average rating": [raw_avg.median(), a.avg_elo.median()],
    "share average rating >= 2900 (%)": [100 * (raw_avg >= 2900).mean(), 100 * (a.avg_elo >= 2900).mean()],
    "share average rating < 1000 (%)": [100 * (raw_avg < 1000).mean(), 100 * (a.avg_elo < 1000).mean()],
    "median rating gap": [raw_gap.median(), a.gap.median()]}, index=["before (all parsed games)", "after (analysis set)"]).round(2)
display(cmp_.T)
say(f"Before cleaning {cmp_.iloc[0]['share < 10 half-moves (%)']:.1f} % of games have fewer than 10 half-moves and {cmp_.iloc[0]['share average rating >= 2900 (%)']:.1f} % an average rating at engine level; "
    f"after cleaning both shares are {cmp_.iloc[1]['share < 10 half-moves (%)']:.1f} % and {cmp_.iloc[1]['share average rating >= 2900 (%)']:.1f} %, while the medians barely move: the rules removed errors without reshaping the data. "
    "(The rating columns before cleaning cover only games in which both PGN ratings are written.)")
del raw, raw_avg, raw_gap, a

,before (all parsed games),after (analysis set)
games,1263916.00,863971.0
median half-moves,77.00,82.0
share < 10 half-moves (%),7.77,0.0
games with both PGN ratings,1005668.00,863971.0
median average rating,2069.00,2092.5
share average rating >= 2900 (%),0.98,0.0
share average rating < 1000 (%),2.91,0.0
median rating gap,164.00,160.0


Before cleaning 7.8 % of games have fewer than 10 half-moves and 1.0 % an average rating at engine level; after cleaning both shares are 0.0 % and 0.0 %, while the medians barely move: the rules removed errors without reshaping the data. (The rating columns before cleaning cover only games in which both PGN ratings are written.)

## 10. Cleaning log

One line per step, in the order the steps were applied: remove steps change the number of games, fix steps change values (rows affected = games or player entries changed), flag steps only add a flag. Written to `reports/cleaning_log.csv`.

In [29]:
parts = [os.path.join(INTERIM, f"_log_{p}.csv") for p in ("4_rules", "7_filters", "8_outliers")]
clog = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
clog.insert(0, "step_no", range(len(clog)))
clog.to_csv(os.path.join(REPORTS, "cleaning_log.csv"), index=False)
display(clog.set_index("step_no"))
removed = clog[clog.action == "remove"]
start, final_n = int(clog.rows_left.iloc[0]), int(clog[clog.stage == "3 analysis filters"].rows_left.iloc[-1])
say(f"**{final_n:,}** of the {start:,} parsed games ({100 * final_n / start:.1f} %) form the analysis set. The largest losses: "
    + ", ".join(f"{r.step.split(' ', 1)[1]} ({r.rows_affected:,})" for r in removed.sort_values("rows_affected", ascending=False).head(3).itertuples()) + ".")

,stage,step,action,rows_affected,rows_left
step_no,,,,,
0,0 collection,all games parsed (notebook 01),start,1263916,1263916
1,1 cleaning rules,1 unfinished result (*),remove,90666,1173250
2,1 cleaning rules,2 forfeit result code,remove,29,1173221
3,1 cleaning rules,3 invalid result string,remove,27,1173194
4,2 fixes,result spelling normalised (e.g. ½-½ -> 1/2-1/2),fix,84,1173194
5,1 cleaning rules,4 non-standard variant / Chess960 / start position,remove,13382,1159812
6,1 cleaning rules,5 no moves / unplayed (incl. forfeits),remove,21465,1138347
7,1 cleaning rules,6 engine-level Elo (2900-4000),remove,8383,1129964
8,1 cleaning rules,7 adjudicated game,remove,1,1129963


**863,971** of the 1,263,916 parsed games (68.4 %) form the analysis set. The largest losses: both Elos known (238,168), unfinished result (*) (90,666), time control known (incl. unresolved glitchy clocks) (23,938).

## 11. Storage

### 11.1 The final table

`clean/analysis_games.parquet` holds the games of the analysis set with all raw columns (for lineage), the repaired fields, the derived columns and the outlier flags: one row per game, key `game_url`.

In [30]:
der = pd.read_parquet(DERIVED); flags = pd.read_parquet(FLAGS)
assert (der.src_row.to_numpy() == flags.src_row.to_numpy()).all() and der.src_row.is_monotonic_increasing
keep = np.zeros(pq.ParquetFile(CLEAN_GAMES).metadata.num_rows, dtype=bool); keep[der.src_row.to_numpy()] = True
extra = pd.concat([der.drop(columns=["src_row", "n_plies", "score_white"]), flags.drop(columns="src_row")], axis=1)
write_rows(CLEAN_GAMES, FINAL, keep, extra)
md_ = pq.ParquetFile(FINAL).metadata
say(f"`{FINAL}`: **{md_.num_rows:,} rows × {md_.num_columns} columns**, {os.path.getsize(FINAL) / 1e6:.0f} MB.")
del der, flags, extra

`..\data\clean\analysis_games.parquet`: **863,971 rows × 90 columns**, 87 MB.

### 11.2 Format choice

| Format | Used for | Reason |
|---|---|---|
| Parquet | the large tables (`all_games`, `clean_games`, `analysis_games`) | compressed, keeps the type of every column (integers, booleans, dates), reads a few columns without loading the rest |
| CSV | small result tables and reports (`cleaning_log.csv`, `data_dictionary.csv`, curves) | human-readable, opened by any tool |
| SQLite | an optional query copy for the dashboard | indexed queries; not the master copy |

The choice is checked on a 200,000-row sample of the analysis columns: write time, size on disk, time to read two columns, and whether the column types survive the round trip.

In [31]:
COLS = ["game_url", "broadcast_name", "game_date", "tc_class", "tc_borderline", "gap", "avg_elo", "white_elo_final", "underdog_color", "underdog_score",
        "n_plies", "year", "has_underdog", "verified"]
full = pd.read_parquet(FINAL, columns=COLS)
s = full.sample(min(200_000, len(full)), random_state=0).reset_index(drop=True); del full
def sql(f):                                                 # open, use and close the SQLite file (Windows cannot delete an open file)
    con = sqlite3.connect(p_sql)
    try: return f(con)
    finally: con.commit(); con.close()
res = []
with tempfile.TemporaryDirectory() as tmp:
    p_pq, p_csv, p_sql = (os.path.join(tmp, f) for f in ("s.parquet", "s.csv", "s.sqlite"))
    for fmt, write, read2, readall, path in (
        ("Parquet", lambda: s.to_parquet(p_pq, index=False), lambda: pd.read_parquet(p_pq, columns=["gap", "tc_class"]), lambda: pd.read_parquet(p_pq), p_pq),
        ("CSV", lambda: s.to_csv(p_csv, index=False), lambda: pd.read_csv(p_csv, usecols=["gap", "tc_class"]), lambda: pd.read_csv(p_csv), p_csv),
        ("SQLite", lambda: sql(lambda con: s.to_sql("games", con, index=False, if_exists="replace")),
         lambda: sql(lambda con: pd.read_sql("select gap, tc_class from games", con)), lambda: sql(lambda con: pd.read_sql("select * from games", con)), p_sql)):
        t0 = time.perf_counter(); write(); t_w = time.perf_counter() - t0
        t0 = time.perf_counter(); read2(); t_r = time.perf_counter() - t0
        back = readall()
        changed = [f"{c} ({kind(s[c])}->{kind(back[c])})" for c in s.columns if kind(s[c]) != kind(back[c])]
        res.append(dict(format=fmt, size_MB=round(os.path.getsize(path) / 1e6, 1), write_s=round(t_w, 2), read_2_columns_s=round(t_r, 3),
                        types_changed=len(changed), which=", ".join(changed) or "none"))
fmt = pd.DataFrame(res).set_index("format"); fmt.to_csv(os.path.join(REPORTS, "format_comparison.csv")); display(fmt)
say(f"Parquet is **{fmt.loc['CSV', 'size_MB'] / fmt.loc['Parquet', 'size_MB']:.1f}× smaller** than CSV and reads two columns **{fmt.loc['CSV', 'read_2_columns_s'] / max(fmt.loc['Parquet', 'read_2_columns_s'], 1e-4):.0f}× faster**, "
    f"and it is the only format that keeps every column type (types changed by CSV: {fmt.loc['CSV', 'which']}; by SQLite: {fmt.loc['SQLite', 'which']}). **Parquet is the master format.**")
del s

,size_MB,write_s,read_2_columns_s,types_changed,which
format,,,,,
Parquet,11.0,0.18,0.020,0,none
CSV,42.6,1.03,0.079,1,game_date (datetime64[us]->string)
SQLite,42.3,0.91,0.240,4,"game_date (datetime64[us]->string), tc_borderline (bool->int64), h..."


Parquet is **3.9× smaller** than CSV and reads two columns **4× faster**, and it is the only format that keeps every column type (types changed by CSV: game_date (datetime64[us]->string); by SQLite: game_date (datetime64[us]->string), tc_borderline (bool->int64), has_underdog (bool->int64), verified (bool->int64)). **Parquet is the master format.**

### 11.3 Data types after reading the file back

The full final table is written to a temporary Parquet file, read back and compared with the original, column by column (in groups of columns, to keep memory small): data type and values must be identical.

In [32]:
names = pq.ParquetFile(FINAL).schema_arrow.names
changed, unequal = [], []
with tempfile.TemporaryDirectory() as tmp:
    for i in range(0, len(names), 12):
        cols = names[i:i + 12]
        d = pd.read_parquet(FINAL, columns=cols)
        p = os.path.join(tmp, "rt.parquet"); d.to_parquet(p, index=False); b = pd.read_parquet(p)
        changed += [c for c in cols if str(d[c].dtype) != str(b[c].dtype)]
        unequal += [c for c in cols if not d[c].equals(b[c])]
rt = (f"Parquet round trip of {FINAL} ({pq.ParquetFile(FINAL).metadata.num_rows:,} rows x {len(names)} columns), run {datetime.date.today()}\n"
      f"columns whose dtype changed: {changed or 'none'}\ncolumns whose values changed: {unequal or 'none'}\n\n"
      + fmt[["types_changed", "which"]].to_string())
open(os.path.join(REPORTS, "dtype_roundtrip.txt"), "w", encoding="utf-8").write(rt)
assert not changed and not unequal, "round trip changed the table"
say(f"All **{len(names)}** columns come back with the same data type and identical values (`reports/dtype_roundtrip.txt`).")

All **90** columns come back with the same data type and identical values (`reports/dtype_roundtrip.txt`).

### 11.4 Data dictionary

One row per column of the final table: data type (as read back), unit, source, meaning, allowed range (the rule the cleaning enforces, where there is one), observed range and number of missing values. The allowed ranges are then checked against the data. Written to `reports/data_dictionary.csv`.

In [33]:
L, F, D = "Lichess broadcast PGN", "FIDE rating list", "derived (notebook 02)"
DERIVED_MEANING = {   # column: (unit, source, meaning, allowed range)
 "base_hat": ("seconds", D, "Base time estimated from the clocks", "30-14400 or missing"),
 "inc_hat": ("seconds", D, "Increment estimated from the clocks (largest clock increase)", "0-600"),
 "score_white": ("points", D, "White's score from the normalised result", {0.0, 0.5, 1.0}),
 "tc_method": ("-", D, "Rule that parsed the TimeControl string (numeric_min, staged, text, missing, ...)", None),
 "tc_base_s": ("seconds", D, "Base time of the time control (string, clock estimate or round consensus)", (1, None)),
 "tc_inc_s": ("seconds", D, "Increment per move", (0, None)),
 "tc_class": ("-", D, "classical / rapid / blitz: base + 60 x increment, <=600 s blitz, <3600 s rapid", set(TC)),
 "tc_source": ("-", D, "How the time control was obtained", None),
 "tc_borderline": ("bool", D, "Total time within 3 minutes of a class threshold (10 or 60 min)", None),
 "period": ("YYYY-MM", D, "Month used for the FIDE lookup: game month, else month of the file", None),
 "date_outside_fide_range": ("bool", D, "Game month outside the downloaded FIDE months", None),
 "white_elo_invalid": ("bool", D, "PGN rating present but outside 1000-2900", None), "black_elo_invalid": ("bool", D, "same for Black", None),
 "white_elo_fide": ("Elo", F, "White's rating in the FIDE list of the month and time control", None), "black_elo_fide": ("Elo", F, "same for Black", None),
 "white_elo_final": ("Elo", D, "White rating used: valid PGN rating, else FIDE list", (1000, 2900)), "black_elo_final": ("Elo", D, "Black rating used (same rule)", (1000, 2900)),
 "white_elo_source": ("-", D, "pgn / fide", {"pgn", "fide"}), "black_elo_source": ("-", D, "pgn / fide", {"pgn", "fide"}),
 "white_elo_mismatch": ("bool", D, "PGN and FIDE ratings differ by more than 100 (False if no FIDE rating)", None), "black_elo_mismatch": ("bool", D, "same for Black", None),
 "analysis_ok": ("bool", D, "Both ratings known", {True}),
 "year": ("year", D, "Year of the game (from period)", (2019, 2026)),
 "gap": ("Elo", D, "Absolute rating difference", (0, 1900)), "avg_elo": ("Elo", D, "Mean of the two ratings", (1000, 2900)),
 "has_underdog": ("bool", D, "gap > 0", None), "underdog_is_white": ("bool", D, "White has the lower rating", None),
 "fav_is_white": ("bool", D, "White has the higher rating", None), "underdog_score": ("points", D, "Underdog's score: 1 win, 0.5 draw, 0 loss", {0.0, 0.5, 1.0}),
 "underdog_color": ("-", D, "Colour of the lower-rated player (white for exact ties)", {"white", "black"}),
 "band": ("-", D, "Band of avg_elo", {"<2200", "2200-2499", "2500+"}), "gap_bin": ("Elo", D, "Lower edge of the 50-point gap bin; 600 = 600 or more", set(range(0, 650, 50))),
 "elo_mismatch_any": ("bool", D, "PGN and FIDE differ by >100 for either player", None),
 "verified": ("bool", D, "Both ratings within 100 points of the FIDE list", None),
 "len_outlier_iqr": ("bool", D, "Game length beyond the IQR fence of its time control", None), "len_outlier_mad": ("bool", D, "Game length robust z > 3.5", None),
 "short_game": ("bool", D, "Fewer than 20 half-moves", None), "iforest_outlier": ("bool", D, "Isolation-Forest outlier (1 % per time control)", None),
}
RAW_ALLOWED = {"n_plies": (10, None), "game_url": "unique", "source_month": None}
UNITS = {"white_elo": "Elo", "black_elo": "Elo", "n_plies": "half-moves", "first_clk_w": "seconds", "first_clk_b": "seconds", "last_clk_w": "seconds",
         "last_clk_b": "seconds", "game_date": "date", "source_month": "YYYY-MM", "utc_date": "date", "date": "date", "utc_time": "time"}
rp = os.path.join(REPORTS, "raw_profile.csv")
raw_meaning = (pd.read_csv(rp).query("table == 'all_games.parquet'").set_index("column")[["source", "meaning"]] if os.path.exists(rp)
               else pd.DataFrame(columns=["source", "meaning"]))
def rng_text(s):
    if pd.api.types.is_bool_dtype(s): return "True / False"
    if pd.api.types.is_numeric_dtype(s): return f"{s.min():.6g} to {s.max():.6g}" if s.notna().any() else "all missing"
    if pd.api.types.is_datetime64_any_dtype(s): return f"{s.min():%Y-%m-%d} to {s.max():%Y-%m-%d}" if s.notna().any() else "all missing"
    u = s.nunique(dropna=True)
    return ", ".join(map(str, sorted(s.dropna().unique())[:8])) if u <= 8 else f"{u:,} distinct values"
def check(s, allowed):
    if allowed is None: return None
    if allowed == "unique": return int(s.duplicated().sum())
    if isinstance(allowed, set): return int((~s.dropna().isin(list(allowed))).sum())
    if isinstance(allowed, tuple):
        lo, hi = allowed; x = s.dropna()
        return int((x < lo).sum() if lo is not None else 0) + int((x > hi).sum() if hi is not None else 0)
    return None
def allowed_text(allowed):
    if allowed is None: return "-"
    if isinstance(allowed, set): return ", ".join(map(str, sorted(allowed, key=str)))
    if isinstance(allowed, tuple): return f"{'' if allowed[0] is None else allowed[0]} to {'' if allowed[1] is None else allowed[1]}".strip()
    return str(allowed)
names = pq.ParquetFile(FINAL).schema_arrow.names
rows = []
for i in range(0, len(names), 12):
    d = pd.read_parquet(FINAL, columns=names[i:i + 12])
    for c in d.columns:
        s = d[c]
        if c in DERIVED_MEANING: unit, src, meaning, allowed = DERIVED_MEANING[c]
        else:
            unit, allowed = UNITS.get(c, "-"), RAW_ALLOWED.get(c)
            src = {"Lichess PGN": L, "parser (this notebook)": "parser (notebook 01)"}.get(raw_meaning.source.get(c, "Lichess PGN"), L)
            meaning = raw_meaning.meaning.get(c, "PGN tag, as written")
        rows.append(dict(column=c, dtype=kind(s), unit=unit, source=src, meaning=meaning, allowed_range=allowed_text(allowed),
                         observed_range=rng_text(s), n_missing=int(s.isna().sum()), violations=check(s, allowed)))
dd = pd.DataFrame(rows)
dd.to_csv(os.path.join(REPORTS, "data_dictionary.csv"), index=False)
bad = dd[dd.violations.fillna(0) > 0]
say(f"`reports/data_dictionary.csv`: **{len(dd)} columns**; {int(dd.violations.notna().sum())} have an allowed range or value set, "
    + ("**all values lie inside it.**" if bad.empty else f"**violations in: {', '.join(bad.column)}** (see table)."))
USED = ["game_url", "broadcast_name", "year", "period", "tc_class", "tc_source", "tc_borderline", "n_plies", "white_elo_final", "black_elo_final", "white_elo_source",
        "white_elo_fide", "gap", "avg_elo", "band", "gap_bin", "underdog_color", "underdog_score", "has_underdog", "score_white", "elo_mismatch_any", "verified", "iforest_outlier"]
display(dd.set_index("column").loc[[c for c in USED if c in set(dd.column)], ["dtype", "unit", "source", "meaning", "allowed_range", "observed_range", "n_missing"]])
if not bad.empty:
    display(bad[["column", "allowed_range", "observed_range", "violations"]])
if "tc_base_s" in set(bad.column):
    tb = pd.read_parquet(FINAL, columns=["time_control", "tc_base_s", "tc_inc_s", "tc_class"]).query("tc_base_s < 1")
    say(f"**Known parser gap.** {len(tb)} games have a base time of 0 s. Their strings are written as clock times (e.g. `{tb.time_control.iloc[0]}`), "
        "that no rule of `parse_tc` reads; their class "
        f"({', '.join(f'{k} {v}' for k, v in tb.tc_class.value_counts().items())}) is therefore probably wrong. Add a rule to `parse_tc` (section 2) for this spelling.")
say(f"The other {len(dd) - len(USED)} columns are raw PGN tags carried along for lineage (title, opening, study, team, clock tags, …); they are described in the CSV.")

`reports/data_dictionary.csv`: **90 columns**; 18 have an allowed range or value set, **all values lie inside it.**

,dtype,unit,source,meaning,allowed_range,observed_range,n_missing
column,,,,,,,
game_url,string,-,Lichess broadcast PGN,Lichess URL of the game; unique key of a game,unique,"863,971 distinct values",0
broadcast_name,string,-,Lichess broadcast PGN,Name of the Lichess broadcast (tournament),-,"9,957 distinct values",0
year,int64,year,derived (notebook 02),Year of the game (from period),2019 to 2026,2019 to 2026,0
period,string,YYYY-MM,derived (notebook 02),"Month used for the FIDE lookup: game month, else month of the file",-,84 distinct values,0
tc_class,string,-,derived (notebook 02),"classical / rapid / blitz: base + 60 x increment, <=600 s blitz, <...","blitz, classical, rapid","blitz, classical, rapid",0
tc_source,string,-,derived (notebook 02),How the time control was obtained,-,"clock_override, clock_raw, clock_snapped, event_name, round_consen...",0
tc_borderline,bool,bool,derived (notebook 02),Total time within 3 minutes of a class threshold (10 or 60 min),-,True / False,0
n_plies,int64,half-moves,parser (notebook 01),Number of half-moves in the move text,10 to,10 to 599,0
white_elo_final,int64,Elo,derived (notebook 02),"White rating used: valid PGN rating, else FIDE list",1000 to 2900,1000 to 2900,0


The other 67 columns are raw PGN tags carried along for lineage (title, opening, study, team, clock tags, …); they are described in the CSV.

## 12. Summary

In [34]:
clog = pd.read_csv(os.path.join(REPORTS, "cleaning_log.csv"))
out = [(p, pq.ParquetFile(p).metadata.num_rows) for p in (ALL_GAMES, CLEAN_GAMES, FINAL)]
display(pd.DataFrame(out, columns=["file", "rows"]).assign(file=lambda d: d.file.map(lambda p: os.path.relpath(p, DATA_ROOT))))
say("Notebook **03_model** starts from `clean/analysis_games.parquet`.")

,file,rows
0,interim\all_games.parquet,1263916
1,interim\clean_games.parquet,1129532
2,clean\analysis_games.parquet,863971


Notebook **03_model** starts from `clean/analysis_games.parquet`.